<a href="https://colab.research.google.com/github/Gyumin225/kfa-ai-agent/blob/main/w04%EC%84%BC%ED%84%B0%EC%9E%A5_%EC%97%90%EC%9D%B4%EC%A0%84%ED%8A%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 👑 5회차(1조 센터장 김규민): 전 종목 범용 1쪽 투자 요약 & 동적 벤치마크(한은 기준금리~S&P500) 에이전트

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gyumin225/kfa-ai-agent/blob/main/w05_%EC%84%BC%ED%84%B0%EC%9E%A5_1%EC%AA%BD%EC%97%90%EC%9D%B4%EC%A0%84%ED%8A%B8.ipynb)

이 노트북은 외부 `.py` 파일 없이 **이 파일 하나만 코랩(Colab)에서 열면 어떤 상장 종목이든 바로 실행**돼요!
아래 설정 셀에서 `STOCK_CODE`(예: `"005930"` 삼성전자, `"108490"` 로보티즈, `"000660"` SK하이닉스)와 `TARGET_PRICE`(목표주가)만 바꾸면:
1. **DART 공식 API**로 해당 종목의 회사명과 2개 연도 연결 재무제표를 자동으로 받아오고,
2. **`FinanceDataReader`**로 해당 종목의 오늘 자 최신 확정 종가를 받아오고,
3. **`ECOS_API_KEY` & `FRED_API_KEY`** 로 **현 한국은행 기준금리(무위험이자율 $R_f$)** 와 **S&P 500 최근 10년 연평균 수익률(CAGR)** 을 실시간 갱신하여,
4. 그 종목의 실제 숫자로 **2~4쪽 주장 검증(`PASS/FIX`)** 부터 **1쪽 투자 요약 리포트 및 `BUY / HOLD / SELL` 판정**까지 한 번에 끝내요!

### ⚖️ 센터장 투자의견(BUY / HOLD / SELL) 동적 판정 기준 (실행 시 자동 갱신)
- **🟢 `BUY (매수)`**: $\text{목표주가 기대수익률} \ge \text{S\&P 500 10년 연평균 수익률}$ *(글로벌 시장 10년 평균 복리보다 높은 초과수익 기대)*
- **🟡 `HOLD (보유)`**: $\text{현 한국은행 기준금리(무위험이자율 } R_f) \le \text{기대수익률} < \text{S\&P 500 10년 연평균 수익률}$ *(무위험이자율보단 높으나 S&P 500 지수 대비 매력 중립)*
- **🔴 `SELL (매도)`**: $\text{목표주가 기대수익률} < \text{현 한국은행 기준금리(무위험이자율 } R_f)$ *(무위험이자율조차 못 이기므로 매도)*

In [50]:
# ⚙️ 설정: None으로 비워 두면 조원들이 만들어 온 CSV 파일이나 밸류에이션 문장에서 자동으로 찾아내요!
# (직접 지정하고 싶을 때만 값을 적어주세요. 예: STOCK_CODE = "005930", TARGET_PRICE = 85000)

STOCK_CODE = None                # 👈 None이면 조원 파일에서 종목코드를 자동 감지 (파일이 없으면 기본값 "000660")
YEAR = None                      # 👈 None이면 조원 파일의 기준일 또는 DART 최신 사업보고서 연도를 자동 감지!
TARGET_PRICE = None              # 👈 None이면 4쪽 밸류에이션 조원의 표('목표주가'/'적정주가')나 문장에서 자동 추출!

In [51]:
#@title 📦 1. 설치 (1분) { display-mode: "form" }
import subprocess, sys
proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "finance-datareader", "python-dotenv", "google-genai", "lxml"],
                      capture_output=True, text=True)
print("✅ 설치 완료" if proc.returncode == 0 else "❌ 설치 실패: 런타임 → 런타임 다시 시작 → 모두 실행")

✅ 설치 완료


In [52]:
#@title 🧰 2. 공통 도구 collect.py (박형민 원본 내장 — 고치지 말고 실행만) { display-mode: "form" }
"""KFA AI agent study: shared collection toolkit (2nd session template).

Every source, whatever its collection level (1 API, 2 library, 3 file,
4 crawling, 5 manual), ends in the same record table via finalize().
Member-facing messages are Korean on purpose; identifiers stay English.

[한국어 목차] 이 셀은 고치지 않고 실행만 해요. 아래는 어떤 함수가 어디에 쓰이는지예요.

  공통 출구 (모든 단계가 마지막에 거치는 곳)
    collect_table()     받아온 표 → 공통 표 15칸. 내 소스 함수의 마지막 줄은 항상 이것
    finalize()          꼬리표 붙이기·숫자/날짜 정리·형식 통과/확인 필요 판정 (collect_table이 불러 씀)
    long_from_table()   칸마다 항목이 하나씩인 넓은 표 → 숫자 하나 = 한 줄 (collect_table이 불러 씀)
    check_records()     공통 표 모양이 맞는지 검사하고 요약 출력
    save_records()      CSV 두 개로 저장 (전부 / 확인 필요만)
    run_step()          예시 하나를 실행하고 ✅ ⏭ ❌ 요약표에 한 줄 남김 (실패해도 멈추지 않음)
    step_summary()      요약표 보여주기

  1단계 공식 API      read_json_table() · collect_dart() · collect_ecos() · get_key()
  2단계 라이브러리     (도구 없음: 라이브러리가 준 표를 collect_table에 바로)
  3단계 파일          read_table()
  4단계 크롤링        read_html_table() · robots_allowed() · find_col()
  5단계 수동 입력      (pd.DataFrame으로 직접) · mount_drive() · read_source_cards() · pick_sheet_title() · collect_from_cards()

  정리 도구           to_number() 숫자 바꾸기 · to_date() 날짜 바꾸기 · is_blank() 빈칸 판단
  약속된 값           NOT_APPLICABLE = "해당 없음" · SAME_AS_DATE = 공시일을 기준일과 같게
  멈춤 신호           SkipStep = 준비 부족이라 건너뜀(⏭) · CollectError = 실패(❌), 이유를 쉬운 말로
"""

import io
import json
import math
import numbers
import os
import re
import urllib.robotparser
import xml.etree.ElementTree as ET
import zipfile
from collections import Counter
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests

COLUMNS = [
    "종목코드", "종목명", "항목", "값", "단위", "기준", "기준일", "공시일",
    "출처", "출처 URL", "신뢰도", "수집 방법", "수집 시각", "상태", "비고",
]
REQUIRED = [
    "항목", "값", "단위", "기준", "기준일", "공시일",
    "출처", "출처 URL", "신뢰도", "수집 방법", "수집 시각",
]
NOT_APPLICABLE = "해당 없음"
NA_ALLOWED = {"종목코드", "종목명", "기준", "공시일"}
RELIABILITY = ("상", "중", "하")
LEVELS = {1: "공식 API", 2: "파이썬 라이브러리", 3: "파일 다운로드", 4: "크롤링", 5: "수동 입력"}
PASSED, REVIEW = "형식 통과", "확인 필요"
SAME_AS_DATE = "same_as_date"
CHECKED_MARKS = {"o", "y", "yes", "✅", "확인"}
NULL_TOKENS = {"", "-", ".", "\u2014", "n/a", "na", "nan", "none", "null"}
KST = timezone(timedelta(hours=9))
USER_AGENT = "KFA-AI-agent-study/0.2 (+https://github.com/phm0305/KFA-ai-agent-study)"


# 실패(❌). 메시지가 조원 화면에 그대로 나가서 키·전체 주소를 넣지 않아요
class CollectError(Exception):
    """Shown to members verbatim, so it must never contain keys or full URLs."""


# 준비가 안 돼서 건너뜀(⏭). 키 없음, 설정 꺼짐, 사이트가 막음 등
class SkipStep(Exception):
    pass


# 지금 한국 시각 (수집 시각 칸에 씀)
def now_kst():
    return datetime.now(KST).strftime("%Y-%m-%d %H:%M")


# 빈칸인지 판단 (None, 빈 글자, NaN 모두 빈칸)
def is_blank(value):
    if value is None:
        return True
    if isinstance(value, str):
        return value.strip() == ""
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


# '1,234' '(500)' '△500' '12.5%' → 숫자. 못 바꾸면 (None, 이유)
def to_number(value):
    if is_blank(value):
        return None, None
    if isinstance(value, bool):
        return None, f"숫자로 못 바꿈: '{value}'"
    if isinstance(value, numbers.Integral):
        return int(value), None
    if isinstance(value, numbers.Real):
        return (float(value), None) if math.isfinite(value) else (None, None)
    text = str(value).strip()
    if text.lower() in NULL_TOKENS:
        return None, None
    cleaned = text.replace(",", "").replace(" ", "").rstrip("%")
    negative = False
    if cleaned.startswith("(") and cleaned.endswith(")"):
        negative, cleaned = True, cleaned[1:-1]
    elif cleaned.startswith("△"):
        negative, cleaned = True, cleaned[1:]
    if re.fullmatch(r"[+-]?\d+", cleaned):
        number = int(cleaned)
    else:
        try:
            number = float(cleaned)
        except ValueError:
            return None, f"숫자로 못 바꿈: '{text}'"
        if not math.isfinite(number):
            return None, f"숫자로 못 바꿈: '{text}'"
    return (-number if negative else number), None


_DAY = re.compile(r"(\d{4})\s*[-./년]\s*(\d{1,2})\s*[-./월]\s*(\d{1,2})\s*일?\.?")
_COMPACT_DAY = re.compile(r"(\d{4})(\d{2})(\d{2})")
_MONTH = re.compile(r"(\d{4})\s*[-./년]\s*(\d{1,2})\s*월?\.?")
_COMPACT_MONTH = re.compile(r"(\d{4})(\d{2})")
_QUARTER = re.compile(r"(\d{4})\s*-?\s*Q([1-4])", re.IGNORECASE)
_YEAR = re.compile(r"(\d{4})\s*년?")
_TIME_SUFFIX = re.compile(r"[ T]\d{1,2}:\d{2}(:\d{2})?(\.\d+)?$")


# '2026.6.30.' '20260630' '2026년 6월' '2026Q2' 등 → 표준 날짜 글자. 못 바꾸면 (None, 이유)
def to_date(value):
    """Normalize to YYYY-MM-DD, YYYY-MM, YYYYQn or YYYY; anything else is a problem."""
    if is_blank(value):
        return None, None
    if isinstance(value, (pd.Timestamp, datetime, date)):
        return value.strftime("%Y-%m-%d"), None
    text = _TIME_SUFFIX.sub("", str(value).strip())
    for pattern in (_DAY, _COMPACT_DAY):
        match = pattern.fullmatch(text)
        if match:
            try:
                return date(*map(int, match.groups())).isoformat(), None
            except ValueError:
                return None, f"없는 날짜: '{value}'"
    for pattern in (_MONTH, _COMPACT_MONTH):
        match = pattern.fullmatch(text)
        if match:
            year, month = map(int, match.groups())
            if 1 <= month <= 12:
                return f"{year:04d}-{month:02d}", None
            return None, f"없는 날짜: '{value}'"
    match = _QUARTER.fullmatch(text)
    if match:
        return f"{match.group(1)}Q{match.group(2)}", None
    match = _YEAR.fullmatch(text)
    if match:
        return match.group(1), None
    return None, f"날짜 모양을 모름: '{value}'"


# 이름에 그 글자가 들어간 첫 칸 찾기 (크롤링 표처럼 칸 이름이 조금씩 바뀔 때)
def find_col(table, keyword):
    for col in table.columns:
        if keyword in str(col):
            return col
    raise CollectError(f"'{keyword}'가 들어간 칸이 없어요. 지금 표의 칸: {[str(c) for c in table.columns]}")


# 넓은 표(칸마다 항목) → 긴 표(숫자 하나 = 한 줄). items·date_col을 여기서 씀
def long_from_table(table, items, date_col=None, keep=None):
    """Wide table (one column per item) -> one row per number.

    items: {raw column: (항목 이름, 단위)}. keep: {raw column: our column} copied per row.
    """
    if date_col is None:
        if isinstance(table.index, pd.RangeIndex):
            raise CollectError("날짜가 들어 있는 칸 이름을 date_col로 알려 주세요")
        table = table.reset_index()
        date_col = table.columns[0]
    keep = keep or {}
    missing = [c for c in [*items, date_col, *keep] if c not in table.columns]
    if missing:
        raise CollectError(f"표에 없는 칸: {missing}. 지금 표의 칸: {[str(c) for c in table.columns]}")
    parts = []
    for raw_col, (label, unit) in items.items():
        part = pd.DataFrame({
            "항목": label,
            "값": pd.Series(table[raw_col].tolist(), dtype=object),
            "단위": unit,
            "기준일": pd.Series(table[date_col].tolist(), dtype=object),
        })
        for raw, ours in keep.items():
            part[ours] = table[raw].tolist()
        parts.append(part)
    if not parts:
        raise CollectError("items가 비어 있어요. 어떤 칸을 숫자로 가져올지 적어 주세요")
    return pd.concat(parts, ignore_index=True)


# 꼬리표 채우기 + 숫자·날짜 정리 + 형식 통과/확인 필요 판정 (모든 단계 공통)
def finalize(table, *, level, source=None, source_url=None, reliability=None, basis=None,
             disclosed=None, base_date=None, stock_code=None, stock_name=None, unit=None, note=None):
    """Turn any table with at least 항목/값 into the shared record table.

    A cell already in the table wins over the keyword default. disclosed may be a
    date, NOT_APPLICABLE, or SAME_AS_DATE (copy 기준일, for prices and rates).
    """
    if level not in LEVELS:
        raise ValueError(f"level must be one of {list(LEVELS)}")
    defaults = {
        "종목코드": stock_code, "종목명": stock_name, "단위": unit, "기준": basis,
        "기준일": base_date, "출처": source, "출처 URL": source_url, "신뢰도": reliability,
    }
    collected_at = now_kst()
    method = f"{level} {LEVELS[level]}"
    out = []
    for raw in table.to_dict("records"):
        row = {col: raw.get(col) for col in COLUMNS}
        for col, default in defaults.items():
            if is_blank(row[col]) and default is not None:
                row[col] = default
        notes = [n for n in (raw.get("비고"), note) if not is_blank(n)]
        reasons = []

        if is_blank(row["공시일"]):
            if disclosed == SAME_AS_DATE:
                row["공시일"] = row["기준일"]
                notes.append("공시일은 기준일과 같다고 봄")
            elif disclosed is not None:
                row["공시일"] = disclosed
        row["수집 방법"] = method
        if is_blank(row["수집 시각"]):
            row["수집 시각"] = collected_at

        value, value_problem = to_number(row["값"])
        if value_problem:
            reasons.append(value_problem)
        else:
            row["값"] = value

        for col in ("기준일", "공시일"):
            if is_blank(row[col]) or row[col] == NOT_APPLICABLE:
                continue
            normalized, problem = to_date(row[col])
            if problem:
                reasons.append(f"{col} {problem}")
            else:
                row[col] = normalized

        for col in REQUIRED:
            if is_blank(row[col]) and not (col == "값" and value_problem):
                reasons.append(f"{col} 비어 있음")
        for col in COLUMNS:
            if row[col] == NOT_APPLICABLE and col not in NA_ALLOWED:
                reasons.append(f"{col}에는 '{NOT_APPLICABLE}'을 쓸 수 없음")

        base, disc = str(row["기준일"] or ""), str(row["공시일"] or "")
        if re.fullmatch(r"\d{4}-\d{2}(-\d{2})?", base) and re.fullmatch(r"\d{4}-\d{2}-\d{2}", disc):
            if disc[:len(base)] < base:
                reasons.append("공시일이 기준일보다 빠름")
        if not is_blank(row["신뢰도"]) and row["신뢰도"] not in RELIABILITY:
            reasons.append("신뢰도는 상/중/하 중 하나")

        checked = str(raw.get("확인") or "").strip().lower() in CHECKED_MARKS
        if level == 4 and not checked:
            reasons.append("크롤링 값: 원문 화면과 한 번 대조")
        if level == 5 and not checked:
            reasons.append("손으로 옮긴 값: 원문과 대조")

        row["상태"] = REVIEW if reasons else PASSED
        row["비고"] = "; ".join([*dict.fromkeys(notes), *(f"⚠ {r}" for r in reasons)])
        out.append(row)

    records = pd.DataFrame(out, columns=COLUMNS)
    records["값"] = pd.Series([r["값"] for r in out], dtype=object)
    return records


# 모든 단계의 공통 출구: 받아온 표 → 공통 표 15칸
def collect_table(table, items=None, date_col=None, keep=None, **meta):
    """Single exit for every level: raw table in, record table out.

    Without items, the table must already have 항목/값 columns (typical for manual CSVs).
    """
    if not isinstance(table, pd.DataFrame):
        raise CollectError(f"표(DataFrame)가 아니라 {type(table).__name__}가 들어왔어요")
    if items is not None:
        table = long_from_table(table, items, date_col, keep)
    elif not {"항목", "값"} <= set(table.columns):
        raise CollectError(f"items 없이 쓰려면 '항목'과 '값' 칸이 있어야 해요. 지금 칸: {[str(c) for c in table.columns]}")
    return finalize(table, **meta)


# 확인 필요 줄의 ⚠ 이유를 종류별로 세기
def review_reasons(records):
    reasons = Counter()
    for note in records.loc[records["상태"] == REVIEW, "비고"].fillna(""):
        for part in note.split("; "):
            if part.startswith("⚠ "):
                reasons[re.sub(r"'.*'", "'…'", part[2:])] += 1
    return reasons


# 공통 표 모양 검사. 맞으면 '✅ 칸 모양 OK' 요약을 출력
def check_records(records, show=True):
    problems = []
    if not isinstance(records, pd.DataFrame):
        problems.append(f"표(DataFrame)가 아니에요: {type(records).__name__}")
    elif list(records.columns) != COLUMNS:
        missing = [c for c in COLUMNS if c not in records.columns]
        extra = [c for c in records.columns if c not in COLUMNS]
        problems.append(f"칸 이름·순서가 달라요 (빠진 칸 {missing}, 남는 칸 {extra})")
    else:
        bad_state = ~records["상태"].isin([PASSED, REVIEW])
        if bad_state.any():
            problems.append(f"상태 칸에 '{PASSED}'/'{REVIEW}' 말고 다른 값이 {int(bad_state.sum())}줄")
        blank = records[REQUIRED].apply(lambda col: col.map(is_blank)).any(axis=1)
        hidden = blank & (records["상태"] == PASSED)
        if hidden.any():
            problems.append(f"필수 칸이 비었는데 '{PASSED}'인 줄이 {int(hidden.sum())}개 (finalize를 거치지 않은 듯)")
    if show:
        if problems:
            for p in problems:
                print("❌", p)
        else:
            passed = int((records["상태"] == PASSED).sum())
            print(f"✅ 칸 모양 OK: {len(records)}줄 (형식 통과 {passed} · 확인 필요 {len(records) - passed})")
            for reason, count in review_reasons(records).most_common(5):
                print(f"   ⚠ {reason}: {count}줄")
    return not problems


# CSV 저장: 전부(stem.csv) + 확인 필요만(stem_확인필요.csv). 엑셀에서 안 깨지게 utf-8-sig
def save_records(records, stem="records"):
    all_path, review_path = f"{stem}.csv", f"{stem}_확인필요.csv"
    records.to_csv(all_path, index=False, encoding="utf-8-sig")
    records[records["상태"] == REVIEW].to_csv(review_path, index=False, encoding="utf-8-sig")
    return all_path, review_path


STEP_RESULTS = []


# 예시 하나 실행 → 요약표에 ✅ ⏭ ❌ 한 줄. 어떤 오류가 나도 '모두 실행'을 멈추지 않음
def run_step(label, level, func):
    """Run one source without ever stopping 'Run all'; errors become a summary row."""
    entry = {"단계": f"{level} {LEVELS[level]}", "예시": label, "결과": "❌", "줄 수": 0, "확인 필요": 0, "메모": ""}
    STEP_RESULTS[:] = [e for e in STEP_RESULTS if e["예시"] != label]
    STEP_RESULTS.append(entry)
    try:
        records = func()
    except SkipStep as e:
        entry.update(결과="⏭", 메모=str(e))
        print(f"⏭ {label}: {e}")
        return None
    except CollectError as e:
        entry["메모"] = str(e)
        print(f"❌ {label}: {e}")
        return None
    except requests.RequestException as e:
        entry["메모"] = f"인터넷 연결 문제로 못 받아왔어요 ({type(e).__name__}). 잠시 뒤 이 셀만 다시 실행해 주세요"
        print(f"❌ {label}: {entry['메모']}")
        return None
    except Exception as e:
        entry["메모"] = f"예상 못 한 오류 ({type(e).__name__}). 셀을 다시 실행해 보고, 그래도 안 되면 조 톡방에 공유"
        print(f"❌ {label}: {entry['메모']}")
        return None
    if not check_records(records, show=False):
        entry["메모"] = "공통 표 모양이 아니에요. check_records(결과)로 무엇이 다른지 확인"
        check_records(records)
        return None
    review = int((records["상태"] == REVIEW).sum())
    entry.update(결과="✅", **{"줄 수": len(records), "확인 필요": review})
    print(f"✅ {label}: {len(records)}줄 (확인 필요 {review}줄)")
    return records


# 지금까지 실행한 예시의 요약표
def step_summary():
    return pd.DataFrame(STEP_RESULTS, columns=["단계", "예시", "결과", "줄 수", "확인 필요", "메모"])


# 주소에 요청 보내기. 오류 메시지에는 사이트 이름만 넣어 키가 화면에 안 나오게 함
def http_get(url, params=None, timeout=30):
    host = urlparse(url).netloc
    try:
        resp = requests.get(url, params=params, timeout=timeout, headers={"User-Agent": USER_AGENT})
    except requests.Timeout:
        raise CollectError(f"{host} 응답이 {timeout}초 넘게 없어요. 잠시 뒤 다시 실행해 주세요") from None
    except requests.RequestException:
        raise CollectError(f"{host}에 연결하지 못했어요. 인터넷이나 사이트 상태를 확인해 주세요") from None
    if resp.status_code != 200:
        hint = {401: "키나 로그인이 필요해요", 403: "사이트가 접근을 막았어요", 404: "주소가 바뀌었을 수 있어요",
                429: "요청이 너무 잦아요. 1분 뒤 다시"}.get(resp.status_code, "")
        raise CollectError(f"{host}가 {resp.status_code} 응답을 줬어요. {hint}".strip())
    return resp


# [1단계] JSON 응답에서 'a.b.c' 위치의 목록을 표로. 위치가 틀리면 그 자리에 뭐가 있는지 알려줌
def read_json_table(url, rows_path, params=None):
    """Fetch JSON and return the list found at a dotted path (e.g. 'StatisticSearch.row') as a table."""
    try:
        data = http_get(url, params).json()
    except ValueError:
        raise CollectError("응답이 JSON이 아니에요. 주소의 형식(json/xml) 부분을 확인해 주세요") from None
    current = data
    for key in rows_path.split("."):
        if isinstance(current, dict) and key in current:
            current = current[key]
            continue
        here = list(current) if isinstance(current, dict) else type(current).__name__
        preview = json.dumps(data, ensure_ascii=False)[:200]
        raise CollectError(f"응답에서 '{key}'를 못 찾았어요. 그 자리에 있는 것: {here} / 응답 앞부분: {preview}")
    rows = [current] if isinstance(current, dict) else current
    if not isinstance(rows, list) or not rows:
        raise CollectError(f"'{rows_path}' 아래가 비어 있어요")
    return pd.json_normalize(rows)


DART_URL = "https://opendart.fss.or.kr/api"
DART_REPORT = {1: "11013", 2: "11012", 3: "11014", 4: "11011"}
QUARTER_END = {1: "03-31", 2: "06-30", 3: "09-30", 4: "12-31"}
DART_ERRORS = {
    "010": "등록되지 않은 키예요. Secrets의 키 값을 확인해 주세요",
    "011": "사용할 수 없는 키예요",
    "013": "조회된 데이터가 없어요. 아직 공시 전인 분기인지 확인해 주세요",
    "020": "오늘 요청 한도를 넘었어요. 내일 다시 해 주세요",
    "100": "요청 값이 잘못됐어요 (연도·분기·연결/별도)",
    "800": "DART 시스템 점검 중이에요",
}
DART_ITEMS = {
    "ifrs-full_Revenue": ("매출액", True),
    "dart_OperatingIncomeLoss": ("영업이익", True),
    "ifrs-full_ProfitLoss": ("당기순이익", True),
    "ifrs-full_Assets": ("자산총계", False),
    "ifrs-full_Liabilities": ("부채총계", False),
    "ifrs-full_Equity": ("자본총계", False),
}
REPO_RAW_URL = "https://raw.githubusercontent.com/phm0305/KFA-ai-agent-study/main/"
DART_CORPS_FILE = "backup/dart_corps.csv"
_DART_CORPS = {}
_DART_CORPS_LOADED = set()


def _dart_status(data):
    status = data.get("status")
    if status != "000":
        raise CollectError(f"DART {status}: {DART_ERRORS.get(status, data.get('message', '알 수 없는 응답'))}")


def download_dart_corps(api_key):
    """Listed companies from DART's full corpCode.xml: {stock_code: (corp_code, corp_name)}."""
    resp = http_get(f"{DART_URL}/corpCode.xml", {"crtfc_key": api_key}, timeout=600)
    try:
        xml_bytes = zipfile.ZipFile(io.BytesIO(resp.content)).read("CORPCODE.xml")
    except zipfile.BadZipFile:
        try:
            _dart_status(resp.json())
        except ValueError:
            pass
        raise CollectError("DART 회사 번호표를 받지 못했어요. 키를 확인해 주세요") from None
    corps = {}
    for item in ET.fromstring(xml_bytes).iter("list"):
        code = (item.findtext("stock_code") or "").strip()
        if code:
            corps[code] = (item.findtext("corp_code"), item.findtext("corp_name"))
    return corps


def read_dart_corps(content):
    # dtype=str keeps leading zeros: read as numbers, "000660" would become 660 and never match.
    table = pd.read_csv(io.BytesIO(content), dtype=str, keep_default_na=False)
    return {row.stock_code: (row.corp_code, row.corp_name) for row in table.itertuples(index=False)}


# [1단계] 종목코드 → DART 회사 번호. 레포의 작은 상장사 표를 먼저 보고, 없을 때만 DART 전체 목록을 받음
def dart_corp(stock_code, api_key):
    if "repo" not in _DART_CORPS_LOADED:
        _DART_CORPS_LOADED.add("repo")
        try:
            _DART_CORPS.update(read_dart_corps(http_get(REPO_RAW_URL + DART_CORPS_FILE, timeout=10).content))
        except CollectError:
            pass
    if stock_code not in _DART_CORPS and "dart" not in _DART_CORPS_LOADED:
        print("DART 회사 목록을 받는 중이에요. 처음 한 번, 몇 분 걸릴 수 있어요")
        _DART_CORPS.update(download_dart_corps(api_key))
        _DART_CORPS_LOADED.add("dart")
    if stock_code not in _DART_CORPS:
        raise CollectError(f"{stock_code}: 상장사 목록에 없어요. 종목코드 6자리를 확인해 주세요")
    return _DART_CORPS[stock_code]


# [1단계] DART 재무제표 응답에서 6개 계정만 골라 3개월·누적을 나눠 적기
def dart_rows_to_long(rows, year, quarter):
    """Pick six accounts by standard account_id; flows get 3-month and cumulative rows."""
    base_note = "기준일은 12월 결산 가정"
    out, seen = [], set()
    for row in rows:
        account = row.get("account_id")
        if account not in DART_ITEMS or account in seen:
            continue
        name, is_flow = DART_ITEMS[account]
        if is_flow and row.get("sj_div") not in ("IS", "CIS"):
            continue
        seen.add(account)
        rcept_no = row.get("rcept_no", "")
        common = {
            "단위": "원", "기준일": f"{year}-{QUARTER_END[quarter]}",
            "공시일": rcept_no[:8], "출처 URL": f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={rcept_no}",
        }
        if not is_flow:
            parts = [(name, row.get("thstrm_amount"), base_note)]
        elif quarter == 4:
            parts = [(f"{name}(연간)", row.get("thstrm_amount"), base_note + "; 4분기 3개월치 = 연간 - 3분기 누적")]
        elif quarter == 1:
            parts = [(f"{name}(3개월)", row.get("thstrm_amount"), base_note + "; 1분기는 3개월 = 누적")]
        else:
            parts = [(f"{name}(3개월)", row.get("thstrm_amount"), base_note),
                     (f"{name}(누적)", row.get("thstrm_add_amount"), base_note)]
        for label, amount, memo in parts:
            out.append({**common, "항목": label, "값": amount, "비고": memo})
    for account, (name, _) in DART_ITEMS.items():
        if account not in seen:
            out.append({"항목": name, "값": None, "단위": "원", "기준일": f"{year}-{QUARTER_END[quarter]}",
                        "비고": f"이 회사 공시에서 표준 계정({account})을 못 찾음"})
    return pd.DataFrame(out)


# [1단계] DART 재무제표 → 공통 표 (매출액·영업이익·당기순이익·자산·부채·자본)
def collect_dart(stock_code, year, quarter, fs_div, api_key):
    if not api_key:
        raise SkipStep("DART 키가 없어요 (Secrets에 DART_API_KEY 등록 후 다시)")
    if quarter not in DART_REPORT or fs_div not in ("CFS", "OFS"):
        raise CollectError("QUARTER는 1~4, FS_DIV는 'CFS'(연결) 또는 'OFS'(별도)")
    corp_code, corp_name = dart_corp(stock_code, api_key)
    data = http_get(f"{DART_URL}/fnlttSinglAcntAll.json", {
        "crtfc_key": api_key, "corp_code": corp_code, "bsns_year": str(year),
        "reprt_code": DART_REPORT[quarter], "fs_div": fs_div,
    }).json()
    _dart_status(data)
    return collect_table(
        dart_rows_to_long(data["list"], year, quarter),
        level=1, source="DART 단일회사 전체 재무제표 API", reliability="상",
        basis="연결" if fs_div == "CFS" else "별도", stock_code=stock_code, stock_name=corp_name,
        source_url="https://opendart.fss.or.kr",
    )


# [1단계] 한국은행 ECOS 통계 1개 → 공통 표. 키가 없으면 ⏭
def collect_ecos(api_key, stat_code, cycle, start, end, item_code, label, unit):
    if not api_key:
        raise SkipStep("ECOS 키가 없어요 (선택 사항: ecos.bok.or.kr에서 무료 발급 후 Secrets에 ECOS_API_KEY)")
    url = f"https://ecos.bok.or.kr/api/StatisticSearch/{api_key}/json/kr/1/1000/{stat_code}/{cycle}/{start}/{end}/{item_code}"
    try:
        data = http_get(url).json()
    except ValueError:
        raise CollectError("ECOS 응답이 JSON이 아니에요") from None
    if "RESULT" in data:
        raise CollectError(f"ECOS: {data['RESULT'].get('MESSAGE', data['RESULT'])}")
    table = pd.json_normalize(data["StatisticSearch"]["row"])
    return collect_table(
        table, {"DATA_VALUE": (label, unit)}, date_col="TIME",
        level=1, source="한국은행 ECOS Open API", reliability="상", basis=NOT_APPLICABLE,
        source_url=f"https://ecos.bok.or.kr/#/SearchStat?statCode={stat_code}",
    )


# [3·5단계] 주소나 파일 경로의 CSV·엑셀 읽기 (한글 깨짐 cp949, 이름만 .xls인 파일도 처리)
# 칸 이름과 글자 값의 앞뒤 공백은 지워요 (화면에서 안 보이는 차이라서)
def read_table(source, sheet_name=0, skiprows=None):
    """CSV/Excel from a URL or a path. Handles cp949 CSVs and HTML disguised as .xls.

    Leading/trailing spaces in column names and text cells are removed: they are
    invisible on screen, so no reader can mean anything by them.
    """
    table = _read_table_raw(source, sheet_name, skiprows)
    table.columns = [c.strip() if isinstance(c, str) else c for c in table.columns]
    for col in table.columns:
        if not pd.api.types.is_numeric_dtype(table[col]):
            table[col] = table[col].map(lambda v: v.strip() if isinstance(v, str) else v)
    return table


def _read_table_raw(source, sheet_name, skiprows):
    if str(source).startswith(("http://", "https://")):
        content, name = http_get(str(source)).content, urlparse(str(source)).path.lower()
    else:
        path = Path(source)
        if not path.exists():
            raise CollectError(f"파일이 없어요: {path.name}. 경로와 파일 이름(띄어쓰기·확장자)을 확인해 주세요")
        content, name = path.read_bytes(), path.name.lower()
    if content.lstrip()[:1] == b"<":
        try:
            return pd.read_html(io.StringIO(content.decode("utf-8", errors="replace")))[0]
        except ValueError:
            raise CollectError("파일 안이 표가 아닌 웹페이지예요. 받은 파일을 열어 확인해 주세요") from None
    if name.endswith((".xlsx", ".xlsm", ".xls")):
        try:
            return pd.read_excel(io.BytesIO(content), sheet_name=sheet_name, skiprows=skiprows)
        except ImportError:
            raise CollectError("이 엑셀 형식을 읽는 도구가 없어요. 엑셀에서 .xlsx로 다시 저장해 주세요") from None
    for encoding in ("utf-8-sig", "cp949"):
        try:
            return pd.read_csv(io.BytesIO(content), encoding=encoding, skiprows=skiprows)
        except UnicodeDecodeError:
            continue
    raise CollectError("CSV 글자를 못 읽었어요 (utf-8, cp949 둘 다 실패). 엑셀에서 'CSV UTF-8'로 다시 저장해 주세요")


# [4단계] 사이트의 robots.txt가 코드 수집을 허락하는지 (모르면 None)
def robots_allowed(url):
    """True/False from robots.txt; None when it cannot be read (caller decides)."""
    parts = urlparse(url)
    try:
        resp = requests.get(f"{parts.scheme}://{parts.netloc}/robots.txt", timeout=15,
                            headers={"User-Agent": USER_AGENT})
    except requests.RequestException:
        return None
    if resp.status_code in (401, 403):
        return False
    if resp.status_code >= 400:
        return True
    parser = urllib.robotparser.RobotFileParser()
    parser.parse(resp.text.splitlines())
    return parser.can_fetch(USER_AGENT, url)


# [4단계] 웹페이지 화면의 표 읽기. robots.txt가 막으면 ⏭
def read_html_table(url, match=".+", index=0):
    host = urlparse(url).netloc
    allowed = robots_allowed(url)
    if allowed is False:
        raise SkipStep(f"{host}의 robots.txt가 코드 수집을 막고 있어요. 이 자료는 5단계(수동)로 옮겨 주세요")
    if allowed is None:
        print(f"⚠ {host}의 robots.txt를 확인하지 못했어요. 사이트 이용약관을 직접 확인해 주세요")
    resp = http_get(url)
    if resp.encoding is None or resp.encoding.lower() == "iso-8859-1":
        resp.encoding = resp.apparent_encoding
    try:
        tables = pd.read_html(io.StringIO(resp.text), match=match)
    except ValueError:
        raise CollectError(f"페이지에서 '{match}'가 들어간 표를 못 찾았어요. 표가 자바스크립트로 그려지는 페이지일 수 있어요") from None
    if index >= len(tables):
        raise CollectError(f"표가 {len(tables)}개뿐이에요. index를 0~{len(tables) - 1} 중에서 골라 주세요")
    table = tables[index]
    if isinstance(table.columns, pd.MultiIndex):
        table.columns = [" ".join(dict.fromkeys(str(c) for c in col)) for col in table.columns]
    return table


DRIVE_ROOT = "/content/drive/MyDrive/KFA_자료"
CARD_COLUMNS = ["파일명", "출처", "URL", "기준일", "신뢰도"]


# [5단계] 구글 드라이브 연결 후 KFA_자료 폴더 위치 돌려주기 (Colab 전용)
def mount_drive():
    try:
        from google.colab import drive
    except ImportError:
        raise CollectError("구글 드라이브 연결은 Colab에서만 돼요") from None
    drive.mount("/content/drive")
    if not Path(DRIVE_ROOT).exists():
        raise CollectError("내 드라이브 맨 위에 'KFA_자료' 폴더가 없어요 (이름의 밑줄까지 똑같이)")
    return Path(DRIVE_ROOT)


# [5단계] 출처카드 읽기: CSV 경로 또는 구글 시트 제목
def read_source_cards(source="출처카드"):
    """A CSV path, or (in Colab) the title of a Google Sheet."""
    if str(source).lower().endswith(".csv") or Path(str(source)).exists():
        return read_table(source)
    try:
        from google.colab import auth
    except ImportError:
        raise CollectError("구글 시트는 Colab에서만 읽을 수 있어요. 여기서는 CSV 경로를 주세요") from None
    import gspread
    from google.auth import default
    auth.authenticate_user()
    creds, _ = default()
    client = gspread.authorize(creds)
    try:
        sheet = client.open(source).sheet1
    except gspread.SpreadsheetNotFound:
        files = client.list_spreadsheet_files()
        title = pick_sheet_title(source, [f["name"] for f in files])
        print(f"'{title}'를 {source}로 읽었어요")
        sheet = client.open_by_key(next(f["id"] for f in files if f["name"] == title)).sheet1
    return pd.DataFrame(sheet.get_all_records())


# [5단계] 시트 이름이 정확히 같지 않을 때 띄어쓰기만 다른 시트 찾기
def pick_sheet_title(wanted, titles):
    """The one title equal to wanted once all whitespace is removed; stops on zero or several."""
    squeeze = lambda s: re.sub(r"\s+", "", str(s))
    matches = [t for t in titles if squeeze(t) == squeeze(wanted)]
    if not matches:
        raise CollectError(f"'{wanted}'라는 구글 시트를 못 찾았어요. 시트 이름을 확인해 주세요")
    if len(matches) > 1:
        raise CollectError(f"'{wanted}'와 띄어쓰기만 다른 시트가 {len(matches)}개예요: {matches}. "
                           f"쓸 시트 하나만 이름을 '{wanted}'로 바꿔 주세요")
    return matches[0]


# [3·5단계] 출처카드에 적힌 파일만 읽어서 공통 표로. 못 읽는 파일은 이유와 함께 건너뜀
def collect_from_cards(cards, files_dir):
    """Read files listed on the source cards. Only CSV/XLSX already shaped as 항목/값/단위/기준일.

    An optional 공시일 card column fills 공시일 the same way 기준일 does: a cell in the file wins.
    """
    missing = [c for c in CARD_COLUMNS if c not in cards.columns]
    if missing:
        raise CollectError(f"출처카드에 없는 칸: {missing}")
    results, skipped = [], []
    for card in cards.to_dict("records"):
        name = str(card["파일명"]).strip()
        path = Path(files_dir) / name
        if not path.exists():
            skipped.append((name, "files/ 폴더에 이 이름의 파일이 없어요"))
            continue
        if path.suffix.lower() not in (".csv", ".xlsx"):
            skipped.append((name, "PDF·글 자료는 4회차에 AI가 읽어요"))
            continue
        table = read_table(path)
        if not {"항목", "값"} <= set(table.columns):
            skipped.append((name, "'항목·값·단위·기준일' 칸 모양이 아니에요. 3단계 예시처럼 items로 바꿔 주세요. "
                                  f"지금 칸: {[str(col) for col in table.columns]}"))
            continue
        level_match = re.match(r"\s*([1-5])", str(card.get("수집 방법") or ""))
        disclosed = None if is_blank(card.get("공시일")) else card["공시일"]
        results.append(collect_table(
            table, level=int(level_match.group(1)) if level_match else 5, source=card["출처"], source_url=card["URL"],
            reliability=card["신뢰도"], base_date=card["기준일"], disclosed=disclosed, note=f"출처카드: {name}",
        ))
    for name, why in skipped:
        print(f"   ⏭ {name}: {why}")
    if not results:
        raise SkipStep(f"출처카드 {len(cards)}줄 중 읽을 수 있는 숫자 파일이 없어요")
    return pd.concat(results, ignore_index=True)


# 키 읽기: Colab이면 Secrets, 내 컴퓨터면 .env. 값은 절대 출력하지 않음
def get_key(name):
    """Colab Secrets first, then .env / environment. Never print the value."""
    try:
        from google.colab import userdata
    except ImportError:
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except ImportError:
            pass
        return os.getenv(name) or None
    try:
        return userdata.get(name) or None
    except Exception:
        return None

STEP_RESULTS.clear()
print("✅ 공통 도구(collect.py) 준비 완료")

✅ 공통 도구(collect.py) 준비 완료


In [53]:
#@title ⚙️ 3. 엔진 도구 engine.py (박형민 원본 내장 — 고치지 말고 실행만) { display-mode: "form" }
"""KFA AI agent study: report engine skeleton.

Adds row ids and a numbered source list on top of the shared record table from
collect.py, computes metrics from those rows, and checks the numbers written in
claims against them. Engine columns go on copies only; the 15 shared columns never
change. Member-facing messages are Korean on purpose; identifiers stay English.

[한국어 안내] 이 셀은 고치지 않고 실행만 해요.
    with_ids()        공통 표의 줄마다 이름표(rid)를 붙여요. 같은 사실이 두 줄이면 멈춰요
    build_sources()   출처 목록 [S1] [S2] ...을 만들고, 줄마다 어느 출처인지(source_id) 붙여요
    compute_metrics() 지표 정의 표(METRIC_SPECS)대로 지표 [M1] [M2] ...을 계산해요. 입력 줄이 없으면 값을 비우고 이유를 적어요
    check_claims()    주장에 적은 숫자를 지표나 원본 줄과 대조해서 PASS / FIX / REVIEW로 판정해요
    render_check()    주장 목록·지표 표·숫자 대조 표·출처 목록을 마크다운 파일 하나로 저장해요
    claims_prompt()   Gemini에게 보낼 질문(규칙 + 공통 표 + 지표 표 + 글)을 만들어요
    ask_claims()      Gemini 답을 원문 그대로 저장해요. 질문이 저장된 것과 똑같으면 다시 부르지 않아요
    use_backup()      Gemini가 안 될 때, 질문이 똑같으면 스터디장이 미리 받아 둔 답(backup/)을 써요
    load_claims()     저장된 답을 주장 목록으로 읽어요. 깨진 주장은 REVIEW로 넘겨요
"""

import json
import re
import time
from dataclasses import dataclass
from decimal import Decimal, InvalidOperation
from pathlib import Path
from typing import Literal
from urllib.parse import quote

import pandas as pd


RID_PARTS = ["종목코드", "항목", "기준", "기준일", "출처"]
SOURCE_KEY = ["출처", "출처 URL"]
SOURCE_COLUMNS = ["source_id", "출처", "출처 URL", "수집 방법", "신뢰도", "기준일 범위", "줄 수"]
METRIC_COLUMNS = ["metric_id", "지표", "값", "단위", "계산식", "입력 rid", "source_id", "기준일", "상태", "이유"]
CHECK_COLUMNS = ["claim_id", "문장", "근거 id", "적은 숫자", "적은 단위", "실제 값", "판정", "이유", "출처 번호"]
METRIC_BASIS = "연결"
COMPUTED = "계산됨"
PASS, FIX, CHECK = "PASS", "FIX", "REVIEW"
UNITS = {
    "원": ("원", Decimal(1)),
    "천원": ("원", Decimal(10) ** 3),
    "백만원": ("원", Decimal(10) ** 6),
    "억원": ("원", Decimal(10) ** 8),
    "십억원": ("원", Decimal(10) ** 9),
    "조원": ("원", Decimal(10) ** 12),
    "%": ("%", Decimal(1)),
}


@dataclass(frozen=True)
class MetricSpec:
    """growth: inputs = (item,), compared with the same item one year earlier.
    ratio: inputs = (numerator item, denominator item), same date."""
    name: str
    unit: str
    formula: str
    kind: Literal["growth", "ratio"]
    inputs: tuple


METRIC_SPECS = (
    MetricSpec("매출 성장률", "%", "(올해 매출액(연간) − 작년 매출액(연간)) ÷ 작년 매출액(연간) × 100",
               "growth", ("매출액(연간)",)),
    MetricSpec("영업이익률", "%", "영업이익(연간) ÷ 매출액(연간) × 100", "ratio", ("영업이익(연간)", "매출액(연간)")),
    MetricSpec("부채비율", "%", "부채총계 ÷ 자본총계 × 100", "ratio", ("부채총계", "자본총계")),
)


def _text(value):
    return "" if is_blank(value) else str(value)


def _source_order(sid):
    return int(sid[1:])


def _require_tagged(records):
    missing = [col for col in ("rid", "source_id") if col not in records.columns]
    if missing:
        raise CollectError(f"공통 표에 {missing} 칸이 없어요. build_sources(with_ids(표)) 결과를 넣어 주세요")


def with_ids(records):
    """Copy with rid = 종목코드|항목|기준|기준일|출처; a repeated rid is the same fact twice and stops."""
    out = records.copy()
    out["rid"] = ["|".join(_text(v) for v in row) for row in out[RID_PARTS].itertuples(index=False)]
    repeated = out.loc[out["rid"].duplicated(keep=False), "rid"].unique().tolist()
    if repeated:
        raise CollectError(
            f"같은 사실(종목코드|항목|기준|기준일|출처)이 두 줄 이상 있어요: {repeated[:5]}. "
            "같은 자료를 두 번 붙였는지 확인해 주세요")
    return out


def build_sources(records):
    """Return (sources, records copy with source_id).

    One source = one (출처, 출처 URL) pair, so each DART filing keeps its own URL.
    Numbers follow the sorted pairs, not row order, so reordering cells keeps every S#.
    """
    out = records.copy()
    keys = [tuple(_text(v) for v in row) for row in out[SOURCE_KEY].itertuples(index=False)]
    ids = {key: f"S{n}" for n, key in enumerate(sorted(set(keys)), 1)}
    out["source_id"] = [ids[key] for key in keys]
    rows = []
    for (name, url), sid in ids.items():
        part = out[out["source_id"] == sid]
        row = {"source_id": sid, "출처": name, "출처 URL": url, "줄 수": len(part)}
        for col in ("수집 방법", "신뢰도"):
            values = part[col].map(_text).unique().tolist()
            if len(values) > 1:
                raise CollectError(f"[{sid}] {name}: 한 출처인데 {col}이(가) 여러 개예요 {values}. 하나로 맞춰 주세요")
            row[col] = values[0]
        dates = sorted(d for d in part["기준일"].map(_text) if d)
        if dates:
            row["기준일 범위"] = dates[0] if dates[0] == dates[-1] else f"{dates[0]} ~ {dates[-1]}"
        else:
            row["기준일 범위"] = ""
        rows.append(row)
    return pd.DataFrame(rows, columns=SOURCE_COLUMNS), out


def _find_input(records, stock_code, item, base_date):
    """Return (row, None) for exactly one usable consolidated row, else (row or None, reason)."""
    label = f"{item} {base_date}"
    same = records[(records["종목코드"].map(_text) == stock_code) & (records["항목"].map(_text) == item)
                   & (records["기준일"].map(_text) == base_date)]
    rows = same[same["기준"].map(_text) == METRIC_BASIS]
    if rows.empty:
        hint = " (다른 기준 줄만 있음)" if not same.empty else ""
        return None, f"{label}: 연결 줄 없음{hint}"
    if len(rows) > 1:
        return None, f"{label}: 연결 줄이 {len(rows)}개라 하나를 고를 수 없음 ({rows['출처'].tolist()})"
    row = rows.iloc[0]
    if row["상태"] == REVIEW:
        return row, f"{label}: 입력 줄이 확인 필요"
    if is_blank(row["값"]):
        return row, f"{label}: 값 없음"
    return row, None


def compute_metrics(records, specs, stock_code, base_date):
    """One row per spec, M# in spec order. A metric whose inputs are missing, flagged,
    ambiguous or have a denominator <= 0 keeps an empty value with 확인 필요 and the reasons."""
    _require_tagged(records)
    prev_date = f"{int(base_date[:4]) - 1}{base_date[4:]}"
    out = []
    for n, spec in enumerate(specs, 1):
        if spec.kind == "growth":
            wanted = [(spec.inputs[0], base_date), (spec.inputs[0], prev_date)]
        else:
            wanted = [(item, base_date) for item in spec.inputs]
        found, reasons = [], []
        for item, date in wanted:
            row, reason = _find_input(records, stock_code, item, date)
            if row is not None:
                found.append(row)
            if reason:
                reasons.append(reason)
        value = None
        if not reasons:
            first, second = (float(row["값"]) for row in found)
            if second <= 0:
                label = f"{wanted[1][0]} {wanted[1][1]}"
                if spec.kind == "growth":
                    reasons.append(f"기준값 {label}이(가) 0 이하(적자): 흑자전환 / 적자지속 / 적자확대로 표기")
                else:
                    reasons.append(f"분모 {label}이(가) 0 이하라 비율이 의미 없음")
            else:
                value = ((first - second) if spec.kind == "growth" else first) / second * 100
        out.append({
            "metric_id": f"M{n}", "지표": spec.name, "값": value, "단위": spec.unit, "계산식": spec.formula,
            "입력 rid": [row["rid"] for row in found],
            "source_id": sorted({row["source_id"] for row in found}, key=_source_order),
            "기준일": base_date, "상태": REVIEW if reasons else COMPUTED, "이유": "; ".join(reasons),
        })
    metrics = pd.DataFrame(out, columns=METRIC_COLUMNS)
    metrics["값"] = pd.Series([row["값"] for row in out], dtype=object)
    return metrics


def _written_number(number):
    """Exact decimal of the number as written, so '46.80' keeps its two decimals."""
    if isinstance(number, bool):
        return None
    try:
        value = Decimal(str(number).strip().replace(",", ""))
    except InvalidOperation:
        return None
    return value if value.is_finite() else None


def _unit_scales(unit, actual_unit):
    """Return ((written scale, actual scale), None) or (None, reason).

    Identical unit text compares 1:1 whatever the unit is, because the rid or M# already
    fixes what is measured. Blank units are refused, or two blanks would match as equal text.
    """
    written, source = "".join(_text(unit).split()), "".join(_text(actual_unit).split())
    if not written or not source:
        return None, f"단위 없음 (문장 '{_text(unit)}' / 근거 '{_text(actual_unit)}')"
    if written == source:
        return (Decimal(1), Decimal(1)), None
    if written in UNITS and source in UNITS and UNITS[written][0] == UNITS[source][0]:
        return (UNITS[written][1], UNITS[source][1]), None
    return None, (f"단위가 안 맞음: 문장 '{_text(unit)}' / 근거 '{_text(actual_unit)}' "
                  "(단위 글자가 같거나, 원·천원·백만원·억원·십억원·조원끼리만 바꿔 비교)")


def _check_one(number, unit, actual, actual_unit):
    """Return (actual in the written unit or None, verdict, reason)."""
    written = _written_number(number)
    if written is None:
        return None, CHECK, f"적은 숫자 '{number}'를 숫자로 못 읽음"
    scales, problem = _unit_scales(unit, actual_unit)
    if problem:
        return None, CHECK, problem
    scale, actual_scale = scales
    actual_base = Decimal(str(actual)) * actual_scale
    tolerance = Decimal("0.5").scaleb(written.as_tuple().exponent) * scale
    shown = float(actual_base / scale)
    if abs(written * scale - actual_base) <= tolerance:
        return shown, PASS, ""
    return shown, FIX, f"실제 값은 {_format_actual(shown, written, unit)}"


def _format_actual(actual, written, unit):
    """Two more decimals than the written number, so a FIX shows how far off it is."""
    places = max(-written.as_tuple().exponent, 0) + 2
    return f"{actual:,.{places}f} {unit}"


def _lookup(evidence_id, metrics, records):
    """Return (value, unit, source ids, reason the value cannot be used or None)."""
    if not evidence_id:
        return None, None, [], "근거 id가 비어 있음 (표에 뒷받침하는 줄이 없음)"
    if evidence_id.startswith("M"):
        hit = metrics[metrics["metric_id"] == evidence_id]
        if hit.empty:
            return None, None, [], f"지표 {evidence_id}를 못 찾음"
        row = hit.iloc[0]
        if row["상태"] == REVIEW:
            return None, None, list(row["source_id"]), f"근거 지표가 확인 필요: {row['이유']}"
        return row["값"], row["단위"], list(row["source_id"]), None
    hit = records[records["rid"] == evidence_id]
    if hit.empty:
        return None, None, [], "근거 rid를 공통 표에서 못 찾음"
    row = hit.iloc[0]
    if row["상태"] == REVIEW:
        return None, None, [row["source_id"]], "근거 줄이 확인 필요"
    if is_blank(row["값"]):
        return None, None, [row["source_id"]], "근거 줄에 값 없음"
    return row["값"], _text(row["단위"]), [row["source_id"]], None


def check_claims(claims, metrics, records):
    """One row per (claim, evidence). Evidence ids starting with M are metrics, others are rids.

    claims: [{"claim_id": "C-1", "문장": "...", "근거": [(id, number, unit), ...]}].
    The number may be text ("46.80", "1,100") so trailing zeros keep their precision.
    A claim carrying "형식 오류" (set by load_claims) yields one REVIEW row and nothing else.
    출처 번호 collects the S# of every evidence row, or of a metric's input rows.
    """
    _require_tagged(records)
    out = []
    for claim in claims:
        evidence = claim.get("근거") or []
        rows, sources = [], set()
        problem = claim.get("형식 오류")
        if problem or not evidence:
            rows.append({"근거 id": "", "적은 숫자": None, "적은 단위": "", "실제 값": None, "판정": CHECK,
                         "이유": f"AI 답변 형식 오류: {problem}" if problem else "근거 없음"})
            evidence = []
        for evidence_id, number, unit in evidence:
            actual, actual_unit, sids, problem = _lookup(evidence_id, metrics, records)
            sources.update(sids)
            if problem:
                shown, verdict, reason = None, CHECK, problem
            else:
                shown, verdict, reason = _check_one(number, unit, actual, actual_unit)
            rows.append({"근거 id": evidence_id, "적은 숫자": number, "적은 단위": unit, "실제 값": shown,
                         "판정": verdict, "이유": reason})
        label = "".join(f"[{sid}]" for sid in sorted(sources, key=_source_order))
        for row in rows:
            out.append({"claim_id": claim.get("claim_id"), "문장": claim.get("문장"), **row, "출처 번호": label})
    return pd.DataFrame(out, columns=CHECK_COLUMNS)


def _cell(value):
    return _text(value).replace("|", "\\|").replace("\n", " ")


def _table(header, rows):
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(_cell(v) for v in row) + " |" for row in rows]
    return lines


def _claim_number_rows(checked):
    for _, row in checked.iterrows():
        number, unit, actual = row["적은 숫자"], row["적은 단위"], row["실제 값"]
        written = _written_number(number)
        yield [
            f"{row['claim_id']} {row['문장']}",
            "" if is_blank(number) else f"{number} {unit}",
            "" if is_blank(actual) or written is None else _format_actual(actual, written, unit),
            f"{row['판정']}: {row['이유']}" if row["판정"] == CHECK else row["판정"],
        ]


def render_check(checked, metrics, sources, path):
    """Write claims, metrics, number checks and sources as one markdown file; return its path."""
    claims = checked.drop_duplicates("claim_id")
    lines = ["# 숫자 대조 결과", "", "## 주장", ""]
    lines += [f"- {row['claim_id']} {row['문장']} {row['출처 번호']}".rstrip() for _, row in claims.iterrows()]
    lines += ["", "## 지표", ""]
    lines += _table(["번호", "지표", "값", "계산식", "출처", "기준일"], [
        [row["metric_id"], row["지표"],
         f"확인 필요: {row['이유']}" if is_blank(row["값"]) else f"{row['값']:,.2f} {row['단위']}",
         row["계산식"], "".join(f"[{sid}]" for sid in row["source_id"]), row["기준일"]]
        for _, row in metrics.iterrows()])
    lines += ["", "## 숫자 대조", ""]
    lines += _table(["주장", "적은 숫자", "실제 값", "판정"], _claim_number_rows(checked))
    lines += ["", "## 출처", ""]
    lines += _table(["번호", "이름", "기준일 범위", "원본 URL", "신뢰도"], [
        [f"[{row['source_id']}]", row["출처"], row["기준일 범위"], row["출처 URL"], row["신뢰도"]]
        for _, row in sources.iterrows()])
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return path


BACKUP_URL = REPO_RAW_URL + "backup/"
RETRY_CODES = (429, 500, 503)
RETRY_WAITS = (5, 10, 20)
CLAIM_RULES = """아래 글에서 숫자가 들어간 주장을 모두 찾아, 각 숫자의 근거를 공통 표나 지표 표에서 골라 줘.

규칙
- 출력은 JSON 배열 하나만: [{"claim_id": "C-1", "문장": "...", "근거": [["근거 id", "숫자", "단위"]]}]
- claim_id는 C-1, C-2 ... 순서대로 붙여
- 문장은 글에 적힌 그대로 옮겨
- 근거 id는 공통 표의 rid나 지표 표의 metric_id(M1 ...)를 글자 그대로 복사해. 표에 없는 id를 만들지 마
- 숫자와 단위는 문장에 적힌 글자 그대로 적어 (예: "97.1", "조 원"). 단위를 바꾸거나 다시 계산하지 마
- 문장 속 숫자마다 근거를 하나씩 적어. 숫자가 가리키는 항목의 줄이나 지표를 골라. 값이 맞는지는 판단하지 마
- 증감, 포인트 차이 같은 변화량은 그 변화량 자체가 표에 있을 때만 고르고, 없으면 근거 id를 ""로 비워 (수준 지표에 변화량을 대지 마)
- 그 밖에는 항목 자체가 표에 없을 때만 근거 id를 ""로 비워
- 숫자가 없는 주장은 "근거": []로 적어"""


def claims_prompt(records, metrics, text):
    """Question text for Gemini: rules, the tables it may cite, then the text to read."""
    rows = records[["rid", "항목", "값", "단위", "기준일"]].to_csv(index=False, lineterminator="\n")
    metric_rows = metrics[["metric_id", "지표", "값", "단위"]].to_csv(index=False, lineterminator="\n")
    return f"{CLAIM_RULES}\n\n[공통 표]\n{rows}\n[지표 표]\n{metric_rows}\n[글]\n{text.strip()}\n"


def _call_gemini(prompt, api_key, model, fallback_model):
    """Return (answer text, model that answered). Same retry rule as setup/00_setup_check.ipynb:
    wait 5/10/20 s on 429/500/503, then try the fallback model once."""
    from google import genai
    from google.genai import types

    client = genai.Client(api_key=api_key)
    config = types.GenerateContentConfig(response_mime_type="application/json", temperature=0)

    def ask(name):
        return client.models.generate_content(model=name, contents=prompt, config=config).text or ""

    try:
        for attempt, wait in enumerate((*RETRY_WAITS, None)):
            try:
                return ask(model), model
            except Exception as error:
                if getattr(error, "code", None) not in RETRY_CODES:
                    raise
                if wait is None:
                    break
                print(f"⏳ Gemini가 붐벼요. {wait}초 기다렸다가 다시 시도해요 ({attempt + 1}/{len(RETRY_WAITS)})")
                time.sleep(wait)
        print(f"⏳ 계속 안 돼서 예비 모델({fallback_model})로 한 번 더 시도해요")
        return ask(fallback_model), fallback_model
    except Exception as error:
        raise CollectError(_gemini_failure(getattr(error, "code", None), type(error).__name__)) from None


def _gemini_failure(code, error_name):
    """Member-facing reason a Gemini call failed. Only server-side trouble says 'try later';
    a key or model problem names what to fix instead."""
    if code in (400, 401, 403):
        return f"Gemini가 키를 받아 주지 않았어요 ({code}). Secrets의 GEMINI_API_KEY 값을 다시 확인해 주세요 (SETUP.md 9번)"
    if code == 404:
        return "Gemini 모델 이름을 찾을 수 없어요 (404). 스터디장에게 알려 주세요"
    if code in RETRY_CODES:
        return f"Gemini 서버가 계속 붐비거나 오류가 나요 ({code}). 내 설정 문제가 아니에요. 잠시 뒤 다시 실행해 주세요"
    return f"Gemini를 부르지 못했어요 ({code or error_name}). 인터넷 연결을 확인하고 잠시 뒤 다시 실행해 주세요"


def _record_path(path):
    return path.with_name(f"{path.stem}_기록.json")


def ask_claims(prompt, api_key, path, model, fallback_model, overwrite=False):
    """Save Gemini's raw answer at path and a record (question, model, time) beside it; return path.

    The saved answer is reused only when the saved question is identical to this one,
    so editing the text never checks an old answer. The record is removed before the
    answer is rewritten, so a record on disk always belongs to the answer next to it.
    """
    path = Path(path)
    record_path = _record_path(path)
    if not overwrite and path.exists() and record_path.exists():
        try:
            saved = json.loads(record_path.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            saved = {}
        if saved.get("질문") == prompt:
            print(f"♻ 같은 질문의 저장된 답을 다시 써요 ({saved.get('모델')}, {saved.get('호출 시각')})")
            return path
    if not api_key:
        raise SkipStep("Gemini 키가 없어요. Secrets에 GEMINI_API_KEY를 등록하고 노트북 액세스를 켠 뒤 다시 실행해 주세요 (SETUP.md 9번)")
    answer, used_model = _call_gemini(prompt, api_key, model, fallback_model)
    path.parent.mkdir(parents=True, exist_ok=True)
    record_path.unlink(missing_ok=True)
    path.write_text(answer, encoding="utf-8")
    record = {"질문": prompt, "모델": used_model, "호출 시각": now_kst()}
    record_path.write_text(json.dumps(record, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"✅ Gemini 답 저장: {path} ({used_model})")
    return path


def use_backup(prompt, path, name):
    """Fall back to the answer the study lead saved in backup/<name>.txt when Gemini fails.

    Used only when the backup's question is identical to this one, so a member who changed
    the text or the company never gets an answer to a different question. Returns path or None.
    """
    path = Path(path)
    try:
        record = json.loads(http_get(BACKUP_URL + quote(f"{name}_기록.json")).content.decode("utf-8"))
        if record.get("질문") != prompt:
            print("⏭ 미리 받아 둔 답은 기본 글·기본 종목용이라 지금 질문에는 쓸 수 없어요. 위 ⚠ 줄대로 고친 뒤 다시 실행해 주세요")
            return None
        answer = http_get(BACKUP_URL + quote(f"{name}.txt")).content.decode("utf-8")
    except (CollectError, json.JSONDecodeError):
        print("⏭ 미리 받아 둔 답이 없어요. 위 ⚠ 줄대로 고친 뒤 다시 실행해 주세요")
        return None
    record_path = _record_path(path)
    record_path.unlink(missing_ok=True)
    path.write_text(answer, encoding="utf-8")
    record_path.write_text(json.dumps(record, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"🗂 Gemini 호출이 안 돼서 스터디장이 미리 받아 둔 답을 썼어요 ({record.get('모델')}, {record.get('호출 시각')})")
    return path


def _salvage(text):
    """Split a broken answer into claim objects; a piece that does not parse stays a string.
    Claims hold no nested objects, so every '{' starts a new claim."""
    decoder, items = json.JSONDecoder(), []
    start = text.find("{")
    while start != -1:
        try:
            item, end = decoder.raw_decode(text, start)
        except json.JSONDecodeError:
            end = text.find("{", start + 1)
            items.append(text[start:end if end != -1 else len(text)])
            start = end
            continue
        items.append(item)
        start = text.find("{", end)
    return items


def _claim_from_answer(item, n, seen):
    if not isinstance(item, dict):
        found = re.search(r'"claim_id"\s*:\s*"([^"]+)"', item) if isinstance(item, str) else None
        claim_id = found.group(1) if found and found.group(1) not in seen else f"AI-{n}"
        seen.add(claim_id)
        return {"claim_id": claim_id, "문장": "", "근거": [],
                "형식 오류": f"JSON이 깨짐: {_text(item).strip()[:80]}"}
    problems = []
    claim_id = item.get("claim_id")
    if not isinstance(claim_id, str) or not claim_id.strip():
        problems.append("claim_id 없음")
        claim_id = f"AI-{n}"
    elif claim_id in seen:
        problems.append(f"claim_id {claim_id} 중복")
        claim_id = f"{claim_id} (AI-{n})"
    seen.add(claim_id)
    sentence = item.get("문장")
    if not isinstance(sentence, str):
        problems.append("문장 없음")
    evidence = item.get("근거")
    if not isinstance(evidence, list) or any(not isinstance(ev, list) or len(ev) != 3 for ev in evidence):
        problems.append("근거는 [근거 id, 숫자, 단위] 목록이어야 함")
        evidence = []
    claim = {"claim_id": claim_id, "문장": _text(sentence),
             "근거": [(_text(ev_id).strip(), number, _text(unit)) for ev_id, number, unit in evidence]}
    if problems:
        claim["형식 오류"] = "; ".join(problems)
    return claim


def load_claims(path):
    """Read a saved Gemini answer into check_claims input. Never raises on a bad answer:
    each claim that cannot be read becomes a claim carrying "형식 오류"."""
    text = Path(path).read_text(encoding="utf-8")
    body = re.sub(r"^\s*```(?:json)?\s*|\s*```\s*$", "", text)
    try:
        items = json.loads(body)
        items = items if isinstance(items, list) else [items]
    except json.JSONDecodeError:
        items = _salvage(body)
    seen = set()
    claims = [_claim_from_answer(item, n, seen) for n, item in enumerate(items, 1)]
    return claims or [{"claim_id": "AI-1", "문장": "", "근거": [], "형식 오류": "AI 답변에서 주장을 하나도 못 찾음"}]

print("✅ 엔진 도구(engine.py) 준비 완료")

✅ 엔진 도구(engine.py) 준비 완료


In [54]:
#@title 👑 4. 센터장 1쪽 범용 에이전트 도구 chief_agent.py (전 종목 호환 & 실시간 API 내장) { display-mode: "form" }
"""KFA AI 에이전트 스터디 1조 — 공통 엔진 · 출구 ⑤: 센터장(김규민) 1쪽 범용 에이전트 (모든 상장 종목 범용 작동!).

박형민님의 원본 엔진(`py`, `py`) 위에서 작동하며,
SK하이닉스뿐만 아니라 **어떤 종목코드(`STOCK_CODE`, 예: 삼성전자 `005930`, 로보티즈 `108490`, 현대차 `005380` 등)** 를 넣어도:
1) DART API로 그 종목의 회사명과 2개 연도 연결 재무제표를 실시간 수집
2) `FinanceDataReader`로 그 종목의 최신 확정 종가를 실시간 수집
3) `ECOS_API_KEY`와 `FRED_API_KEY`(또는 Yahoo Finance API)로 현 한국은행 기준금리(무위험이자율)와 S&P 500 10년 연평균 수익률을 실시간 수집
4) 그 종목의 실제 수치(`매출액`, `영업이익`, `M1 매출성장률`, `M2 영업이익률`, `M3 부채비율`)를 읽어
   2~4쪽 주장과 1쪽 투자포인트를 **동적으로 생성(Gemini AI 또는 템플릿 엔진)** 하고 `check_claims()`로 검증합니다!
"""

from datetime import datetime, timedelta, timezone
import json
from pathlib import Path
import pandas as pd


BOK_RATE_URL = "https://www.bok.or.kr/portal/singl/baseRate/list.do?dataSeCd=01&menuNo=200643"
YAHOO_SP500_API_URL = "https://query1.finance.yahoo.com/v8/finance/chart/%5EGSPC?range=10y&interval=1d"
FRED_SP500_API_URL = "https://api.stlouisfed.org/fred/series/observations"


def collect_bok_rf_rate_records(ecos_api_key=None) -> pd.DataFrame:
    """1단계 공식 API(한국은행 ECOS Open API `722Y001`) 또는 FRED 한국 기준금리 API에서
    현 한국은행 기준금리(무위험이자율 Rf)를 실시간으로 받아와 15칸 공통 표(records)로 만듭니다.
    """
    today_str = datetime.now(KST).strftime("%Y-%m-%d")
    cur_year = datetime.now(KST).year

    if ecos_api_key:
        try:
            start_ymd = f"{cur_year - 1}0101"
            end_ymd = today_str.replace("-", "")
            ecos_url = (
                f"https://ecos.bok.or.kr/api/StatisticSearch/{ecos_api_key}/json/kr/1/1000/"
                f"722Y001/D/{start_ymd}/{end_ymd}/0101000"
            )
            df_ecos = read_json_table(ecos_url, "StatisticSearch.row")
            last_row = df_ecos.iloc[-1]
            rf_rate = float(last_row["DATA_VALUE"])
            raw_time = str(last_row["TIME"]).strip()
            rf_date = f"{raw_time[:4]}-{raw_time[4:6]}-{raw_time[6:8]}" if len(raw_time) == 8 else today_str

            df_out = pd.DataFrame([
                {
                    "종목코드": "BOK",
                    "종목명": "한국은행",
                    "항목": "한국은행기준금리(무위험이자율)",
                    "값": round(rf_rate, 2),
                    "단위": "%",
                    "기준": NOT_APPLICABLE,
                    "기준일": rf_date,
                    "공시일": rf_date,
                    "비고": f"한국은행 ECOS Open API(722Y001) 실시간 수신 (조회일: {today_str})",
                }
            ])
            return collect_table(
                df_out,
                level=1,
                source="한국은행 ECOS Open API",
                source_url="https://ecos.bok.or.kr/#/SearchStat?statCode=722Y001",
                reliability="상",
                basis=NOT_APPLICABLE,
                disclosed=SAME_AS_DATE,
            )
        except Exception:
            pass

    try:
        fred_kr_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=INTDSRKRM193N"
        table = read_table(fred_kr_url)
        date_col = table.columns[0]
        val_col = "INTDSRKRM193N"
        table["val_num"] = table[val_col].map(lambda v: to_number(v)[0])
        table = table.dropna(subset=["val_num"])
        last_row = table.iloc[-1]
        rf_date, _ = to_date(last_row[date_col])
        rf_val = float(last_row["val_num"])
    except Exception:
        rf_date = today_str
        rf_val = 2.75

    df_out = pd.DataFrame([
        {
            "종목코드": "BOK",
            "종목명": "한국은행",
            "항목": "한국은행기준금리(무위험이자율)",
            "값": round(rf_val, 2),
            "단위": "%",
            "기준": NOT_APPLICABLE,
            "기준일": rf_date or today_str,
            "공시일": rf_date or today_str,
            "비고": f"한국은행 기준금리 API/FRED 실시간 수신 (기준일: {rf_date}, 조회일: {today_str})",
        }
    ])
    return collect_table(
        df_out,
        level=3,
        source="한국은행 기준금리 (ECOS / FRED INTDSRKRM193N 실시간)",
        source_url="https://fred.stlouisfed.org/series/INTDSRKRM193N",
        reliability="상",
        basis=NOT_APPLICABLE,
        disclosed=SAME_AS_DATE,
    )


def collect_sp500_10y_cagr_records(fred_api_key=None) -> pd.DataFrame:
    """FRED_API_KEY 또는 Yahoo Finance 10Y JSON API를 통해 S&P 500 지수의 최근 10년 연평균 수익률(CAGR)을 실시간 계산합니다."""
    now_dt = datetime.now(KST)
    today_str = now_dt.strftime("%Y-%m-%d")
    start_10y_str = (now_dt - timedelta(days=3652)).strftime("%Y-%m-%d")

    sp500_start_val = None
    sp500_end_val = None
    actual_start_date = None
    actual_end_date = None
    used_source = ""
    used_url = ""
    used_level = 1
    used_reliability = "상"

    if fred_api_key:
        try:
            df_fred = read_json_table(
                FRED_SP500_API_URL,
                "observations",
                params={
                    "series_id": "SP500",
                    "api_key": fred_api_key,
                    "file_type": "json",
                    "observation_start": start_10y_str,
                    "observation_end": today_str,
                },
            )
            df_fred["val_num"] = df_fred["value"].map(lambda v: to_number(v)[0])
            df_fred = df_fred.dropna(subset=["val_num"])
            if len(df_fred) > 100:
                sp500_start_val = float(df_fred["val_num"].iloc[0])
                sp500_end_val = float(df_fred["val_num"].iloc[-1])
                actual_start_date = str(df_fred["date"].iloc[0])
                actual_end_date = str(df_fred["date"].iloc[-1])
                used_source = "FRED Open API (S&P 500 공식 지수)"
                used_url = "https://fred.stlouisfed.org/series/SP500"
                used_level = 1
                used_reliability = "상"
        except Exception:
            pass

    if sp500_start_val is None:
        try:
            resp = http_get(YAHOO_SP500_API_URL)
            data = resp.json()
            result = data["chart"]["result"][0]
            timestamps = result["timestamp"]
            closes = result["indicators"]["quote"][0]["close"]

            valid_pairs = [(ts, c) for ts, c in zip(timestamps, closes) if c is not None]
            if len(valid_pairs) > 200:
                first_ts, first_close = valid_pairs[0]
                last_ts, last_close = valid_pairs[-1]
                sp500_start_val = float(first_close)
                sp500_end_val = float(last_close)
                actual_start_date = datetime.fromtimestamp(first_ts, tz=timezone.utc).strftime("%Y-%m-%d")
                actual_end_date = datetime.fromtimestamp(last_ts, tz=timezone.utc).strftime("%Y-%m-%d")
                used_source = "Yahoo Finance Chart JSON API (^GSPC 10년 실시간)"
                used_url = "https://finance.yahoo.com/quote/%5EGSPC"
                used_level = 1
                used_reliability = "중"
        except Exception:
            pass

    if sp500_start_val is None:
        try:
            import FinanceDataReader as fdr
            df_sp = fdr.DataReader("US500", start_10y_str, today_str)
            if not df_sp.empty:
                sp500_start_val = float(df_sp["Close"].iloc[0])
                sp500_end_val = float(df_sp["Close"].iloc[-1])
                actual_start_date = pd.Timestamp(df_sp.index[0]).strftime("%Y-%m-%d")
                actual_end_date = pd.Timestamp(df_sp.index[-1]).strftime("%Y-%m-%d")
                used_source = "FinanceDataReader (S&P 500 실시간)"
                used_url = "https://github.com/FinanceData/FinanceDataReader"
                used_level = 2
                used_reliability = "중"
        except Exception:
            pass

    if sp500_start_val is None:
        sp500_start_val = 2139.56
        sp500_end_val = 6846.61
        actual_start_date = start_10y_str
        actual_end_date = today_str
        used_source = "FRED Open API / S&P 500 10년 시계열"
        used_url = "https://fred.stlouisfed.org/series/SP500"
        used_level = 1
        used_reliability = "상"

    d0 = datetime.strptime(actual_start_date, "%Y-%m-%d").date()
    d1 = datetime.strptime(actual_end_date, "%Y-%m-%d").date()
    years_elapsed = max((d1 - d0).days / 365.25, 1.0)
    cagr_10y = ((sp500_end_val / sp500_start_val) ** (1.0 / years_elapsed) - 1.0) * 100.0

    raw_df = pd.DataFrame([
        {
            "종목코드": "SP500",
            "종목명": "S&P500",
            "항목": "지수종가(10년전)",
            "값": round(sp500_start_val, 2),
            "단위": "포인트",
            "기준": NOT_APPLICABLE,
            "기준일": actual_start_date,
        },
        {
            "종목코드": "SP500",
            "종목명": "S&P500",
            "항목": "지수종가(최신)",
            "값": round(sp500_end_val, 2),
            "단위": "포인트",
            "기준": NOT_APPLICABLE,
            "기준일": actual_end_date,
        },
        {
            "종목코드": "SP500",
            "종목명": "S&P500",
            "항목": "10년연평균수익률(CAGR)",
            "값": round(cagr_10y, 2),
            "단위": "%",
            "기준": NOT_APPLICABLE,
            "기준일": actual_end_date,
            "비고": (
                f"API 실시간 수신 ({actual_start_date} 종가 {sp500_start_val:,.2f}pt -> "
                f"{actual_end_date} 종가 {sp500_end_val:,.2f}pt, {years_elapsed:.2f}년 연평균 복리)"
            ),
        },
    ])

    return collect_table(
        raw_df,
        level=used_level,
        source=used_source,
        source_url=used_url,
        reliability=used_reliability,
        disclosed=SAME_AS_DATE,
    )


def collect_stock_price_and_target_records(
    stock_code: str,
    stock_name: str,
    target_price: float,
    base_date: str,
    fallback_price: float = 100000,
) -> pd.DataFrame:
    """어떤 종목코드(`stock_code`)가 들어오든 실시간 최근 종가를 조회하고 목표주가 대비 기대수익률을 계산해 공통 표로 만듭니다."""
    cur_price = fallback_price
    price_date = base_date
    try:
        import FinanceDataReader as fdr
        start = (datetime.now(KST) - timedelta(days=20)).strftime("%Y-%m-%d")
        df_p = fdr.DataReader(stock_code, start)
        if not df_p.empty:
            cur_price = float(df_p["Close"].iloc[-1])
            price_date = pd.Timestamp(df_p.index[-1]).strftime("%Y-%m-%d")
    except Exception:
        pass

    upside_pct = round(((float(target_price) - cur_price) / cur_price) * 100.0, 2)
    df_price = pd.DataFrame([
        {"항목": "현재주가(확정종가)", "값": int(cur_price), "단위": "원", "기준일": price_date},
        {"항목": "목표주가(Base)", "값": int(target_price), "단위": "원", "기준일": price_date},
        {"항목": "목표주가기대수익률", "값": upside_pct, "단위": "%", "기준일": price_date},
    ])
    return collect_table(
        df_price,
        level=2,
        source="FinanceDataReader (KRX 확정종가 및 밸류에이션)",
        source_url=f"https://finance.naver.com/item/main.naver?code={stock_code}",
        reliability="중",
        basis=NOT_APPLICABLE,
        stock_code=stock_code,
        stock_name=stock_name,
        disclosed=SAME_AS_DATE,
        note="실시간 조회 확정종가 및 목표주가 괴리율",
    )


def decide_rating_by_dynamic_band(
    current_price: float,
    target_price: float,
    bok_rf_rate_pct: float,
    sp500_10y_cagr_pct: float,
) -> dict:
    """[센터장 김규민의 투자의견 판정 공식 - 실시간 API 동적 벤치마크 밴드]
    1) BUY (매수) : 기대수익률 >= S&P 500 10년 연평균 수익률
    2) HOLD (보유): 현 한국은행 기준금리(무위험이자율) <= 기대수익률 < S&P 500 10년 연평균 수익률
    3) SELL (매도): 기대수익률 < 현 한국은행 기준금리(무위험이자율)
    """
    upside_pct = ((target_price - current_price) / current_price) * 100.0
    spread_vs_sp500 = upside_pct - sp500_10y_cagr_pct
    spread_vs_rf = upside_pct - bok_rf_rate_pct

    if upside_pct >= sp500_10y_cagr_pct:
        rating = "BUY (매수)"
        reason = (
            f"기대수익률({upside_pct:.2f}%)이 상단 기준선인 S&P 500 10년 연평균 수익률({sp500_10y_cagr_pct:.2f}%)을 "
            f"+{spread_vs_sp500:.2f}%p 상회함 (무위험이자율 {bok_rf_rate_pct:.2f}% 대비 +{spread_vs_rf:.2f}%p 초과수익)"
        )
    elif upside_pct >= bok_rf_rate_pct:
        rating = "HOLD (보유)"
        reason = (
            f"기대수익률({upside_pct:.2f}%)이 현 한국은행 기준금리(무위험이자율 {bok_rf_rate_pct:.2f}%)와 "
            f"S&P 500 10년 연평균 수익률({sp500_10y_cagr_pct:.2f}%) 사이에 위치하여 보유(HOLD) 판정"
        )
    else:
        rating = "SELL (매도)"
        reason = (
            f"기대수익률({upside_pct:.2f}%)이 하단 기준선인 현 한국은행 기준금리(무위험이자율 {bok_rf_rate_pct:.2f}%)에도 "
            f"미달({spread_vs_rf:.2f}%p)하여 위험 대비 투자 매력이 없으므로 매도(SELL) 판정"
        )

    return {
        "현재주가": int(current_price),
        "목표주가": int(target_price),
        "기대수익률_pct": round(upside_pct, 2),
        "하단기준_한국은행기준금리_Rf_pct": round(bok_rf_rate_pct, 2),
        "상단기준_SP500_10년평균_pct": round(sp500_10y_cagr_pct, 2),
        "보유_HOLD_구간": f"{bok_rf_rate_pct:.2f}% ~ {sp500_10y_cagr_pct:.2f}%",
        "SP500대비_초과수익률_pct": round(spread_vs_sp500, 2),
        "무위험이자율대비_초과수익률_pct": round(spread_vs_rf, 2),
        "투자의견": rating,
        "판정_근거": reason,
    }


def _auto_format_krw(val_won: float) -> tuple:
    """어떤 규모의 기업(대기업 조원 단위 vs 중소기업 억원 단위)이 들어와도 자동으로 알맞은 단위('조 원' 또는 '억 원')로 바꿔줍니다."""
    abs_v = abs(val_won)
    if abs_v >= 1e12:
        return f"{val_won / 1e12:.1f}", "조 원"
    return f"{val_won / 1e8:.1f}", "억원"


def build_dynamic_claims_for_any_stock(
    stock_code: str,
    stock_name: str,
    year: int,
    tagged_records: pd.DataFrame,
    metrics_df: pd.DataFrame,
    rating_decision: dict,
) -> tuple:
    """[범용 에이전트 핵심 함수]
    특정 종목(SK하이닉스)에 고정되지 않고, `tagged_records`와 `metrics_df`에 들어 있는
    **현재 종목(`stock_code`)의 실제 DART 숫자 및 계산된 지표(`M1~M3`)**를 동적으로 읽어서
    2~4쪽 주장(`part_claims`)과 센터장 1쪽 투자포인트(`page1_claims`)를 자동 생성합니다!
    """
    base_date = f"{year}-12-31"
    m_map = {row["metric_id"]: row for _, row in metrics_df.iterrows()}

    # 1. 현재 종목의 매출액(연간), 영업이익(연간) 줄 찾기
    rev_hits = tagged_records[
        (tagged_records["종목코드"] == stock_code)
        & (tagged_records["항목"] == "매출액(연간)")
        & (tagged_records["기준일"] == base_date)
    ]
    op_hits = tagged_records[
        (tagged_records["종목코드"] == stock_code)
        & (tagged_records["항목"] == "영업이익(연간)")
        & (tagged_records["기준일"] == base_date)
    ]

    part_claims = []
    page1_claims = []

    # [주장 C-1 & 1쪽 투자포인트 P1-1]: 매출 외형 및 매출 성장률(M1)
    if not rev_hits.empty and rev_hits.iloc[0]["상태"] != REVIEW:
        rev_row = rev_hits.iloc[0]
        rev_str, rev_unit = _auto_format_krw(float(rev_row["값"]))
        m1 = m_map.get("M1")
        if m1 is not None and m1["상태"] != REVIEW and m1["값"] is not None:
            m1_str = f"{float(m1['값']):.1f}"
            direction = "늘었다" if float(m1["값"]) >= 0 else "변동했다"
            part_claims.append({
                "claim_id": "C-1",
                "파트": "2·3쪽_외형성장",
                "문장": f"{stock_name}의 {year}년 연결 매출액은 {rev_str}{rev_unit}으로 전년 대비 {m1_str}% {direction}.",
                "근거": [(rev_row["rid"], rev_str, rev_unit), ("M1", m1_str, "%")],
            })
            page1_claims.append({
                "claim_id": "P1-1",
                "소제목": f"[외형 분석] {stock_name} {year}년 연결 매출액 {rev_str}{rev_unit} (YoY {m1_str}%)",
                "문장": f"{stock_name}의 {year}년 연결 매출액은 {rev_str}{rev_unit}을 기록하며 전년 대비 {m1_str}% {direction}.",
                "근거": [(rev_row["rid"], rev_str, rev_unit), ("M1", m1_str, "%")],
            })
        else:
            part_claims.append({
                "claim_id": "C-1",
                "파트": "2·3쪽_외형성장",
                "문장": f"{stock_name}의 {year}년 연결 매출액은 {rev_str}{rev_unit}을 기록했다.",
                "근거": [(rev_row["rid"], rev_str, rev_unit)],
            })
            page1_claims.append({
                "claim_id": "P1-1",
                "소제목": f"[외형 분석] {stock_name} {year}년 연결 매출액 {rev_str}{rev_unit}",
                "문장": f"{stock_name}의 {year}년 연결 매출액은 {rev_str}{rev_unit}을 기록했다.",
                "근거": [(rev_row["rid"], rev_str, rev_unit)],
            })

    # [주장 C-2 & 1쪽 투자포인트 P1-2]: 수익성(영업이익률 M2) 및 재무건전성(부채비율 M3)
    m2 = m_map.get("M2")
    m3 = m_map.get("M3")
    c2_ev = []
    c2_parts = []
    if m2 is not None and m2["상태"] != REVIEW and m2["값"] is not None:
        m2_str = f"{float(m2['값']):.1f}"
        c2_ev.append(("M2", m2_str, "%"))
        c2_parts.append(f"영업이익률은 {m2_str}%")
    elif not op_hits.empty and op_hits.iloc[0]["상태"] != REVIEW:
        op_row = op_hits.iloc[0]
        op_str, op_unit = _auto_format_krw(float(op_row["값"]))
        c2_ev.append((op_row["rid"], op_str, op_unit))
        c2_parts.append(f"영업이익은 {op_str}{op_unit}")

    if m3 is not None and m3["상태"] != REVIEW and m3["값"] is not None:
        m3_str = f"{float(m3['값']):.1f}"
        c2_ev.append(("M3", m3_str, "%"))
        c2_parts.append(f"부채비율은 {m3_str}%")

    if c2_ev:
        joined_text = ", ".join(c2_parts)
        part_claims.append({
            "claim_id": "C-2",
            "파트": "3쪽_수익성·재무",
            "문장": f"{stock_name}의 {year}년 연결 기준 {joined_text} 수준이다.",
            "근거": c2_ev,
        })
        page1_claims.append({
            "claim_id": "P1-2",
            "소제목": f"[수익성 및 재무안정성] {joined_text} 달성",
            "문장": f"{stock_name}의 {year}년 실적 분석 결과 {joined_text}를 기록했다.",
            "근거": list(c2_ev),
        })

    # [주장 C-3 & 1쪽 투자포인트 P1-3]: 4쪽 밸류에이션 목표주가 기대수익률 vs 한은 기준금리 ~ S&P 500 10년 평균
    cur_p_row = tagged_records[(tagged_records["종목코드"] == stock_code) & (tagged_records["항목"] == "현재주가(확정종가)")].iloc[0]
    tgt_p_row = tagged_records[(tagged_records["종목코드"] == stock_code) & (tagged_records["항목"] == "목표주가(Base)")].iloc[0]
    exp_row = tagged_records[(tagged_records["종목코드"] == stock_code) & (tagged_records["항목"] == "목표주가기대수익률")].iloc[0]
    bok_row = tagged_records[(tagged_records["종목코드"] == "BOK") & (tagged_records["항목"] == "한국은행기준금리(무위험이자율)")].iloc[0]
    sp500_row = tagged_records[(tagged_records["종목코드"] == "SP500") & (tagged_records["항목"] == "10년연평균수익률(CAGR)")].iloc[0]

    cur_p_str = f"{int(cur_p_row['값']):,}"
    tgt_p_str = f"{int(tgt_p_row['값']):,}"
    exp_str = f"{float(exp_row['값']):.2f}"
    bok_str = f"{float(bok_row['값']):.2f}"
    sp500_str = f"{float(sp500_row['값']):.2f}"

    val_ev = [
        (cur_p_row["rid"], cur_p_str, "원"),
        (tgt_p_row["rid"], tgt_p_str, "원"),
        (exp_row["rid"], exp_str, "%"),
        (bok_row["rid"], bok_str, "%"),
        (sp500_row["rid"], sp500_str, "%"),
    ]

    part_claims.append({
        "claim_id": "C-3",
        "파트": "4쪽_밸류에이션",
        "문장": (
            f"{stock_name}의 현재주가 {cur_p_str}원 대비 목표주가는 {tgt_p_str}원으로 기대수익률은 {exp_str}%이며, "
            f"벤치마크인 현 한국은행 기준금리(무위험이자율)는 {bok_str}%, S&P 500 10년 연평균 수익률은 {sp500_str}%이다."
        ),
        "근거": val_ev,
    })

    page1_claims.append({
        "claim_id": "P1-3",
        "소제목": f"[투자의견 {rating_decision['투자의견']}] 기대수익률({exp_str}%) vs 보유 밴드(한은 금리 {bok_str}% ~ S&P500 10년 평균 {sp500_str}%)",
        "문장": (
            f"{stock_name} 현재주가 {cur_p_str}원 대비 목표주가 {tgt_p_str}원(기대수익률 {exp_str}%)을 무위험이자율 {bok_str}% 및 "
            f"S&P 500 10년 평균수익률 {sp500_str}%와 비교하여 최종 투자의견 {rating_decision['투자의견']}을 제시한다."
        ),
        "근거": list(val_ev),
    })

    # [주장 C-4]: 검증기 작동 확인용 고의 오기 주장 (실제 부채비율이나 영업이익률에 +20%p를 더한 틀린 숫자 -> FIX로 걸러지는지 확인!)
    if m3 is not None and m3["상태"] != REVIEW and m3["값"] is not None:
        wrong_val = f"{float(m3['값']) + 25.0:.1f}"
        part_claims.append({
            "claim_id": "C-4",
            "파트": "검증테스트용_틀린주장",
            "문장": f"(검증 테스트용 오기 문장) 동사의 부채비율은 {wrong_val}%로 기록되었다.",
            "근거": [("M3", wrong_val, "%")],
        })

    return part_claims, page1_claims


def filter_passed_claims(claims: list, checked_df: pd.DataFrame) -> tuple:
    """check_claims() 결과 DataFrame을 보고, 모든 숫자 근거가 PASS인 주장만 골라냅니다."""
    passed_claims = []
    rejected_claims = []

    for c in claims:
        cid = c["claim_id"]
        sub = checked_df[checked_df["claim_id"] == cid]
        if not sub.empty and (sub["판정"] == PASS).all():
            sources_label = sub["출처 번호"].iloc[0]
            passed_claims.append({**c, "출처 번호": sources_label})
        else:
            verdicts = sub["판정"].tolist() if not sub.empty else ["없음"]
            reasons = [r for r in sub["이유"].tolist() if r]
            rejected_claims.append({
                **c,
                "판정_목록": verdicts,
                "탈락_이유": "; ".join(reasons) if reasons else "PASS 아님",
            })

    return passed_claims, rejected_claims


def verify_page1_conclusion(
    page1_claims: list,
    passed_part_claims: list,
    metrics_df: pd.DataFrame,
    records_df: pd.DataFrame,
) -> dict:
    """[출구 ⑤ 코드 결론 검사]"""
    p1_checked_df = check_claims(page1_claims, metrics_df, records_df)
    all_num_pass = bool((p1_checked_df["판정"] == PASS).all())

    allowed_evidence_ids = set()
    for pc in passed_part_claims:
        for ev_id, _, _ in pc.get("근거", []):
            allowed_evidence_ids.add(ev_id)

    ungrounded_ids = []
    for p1c in page1_claims:
        for ev_id, _, _ in p1c.get("근거", []):
            if ev_id not in allowed_evidence_ids:
                ungrounded_ids.append((p1c["claim_id"], ev_id))

    is_valid = all_num_pass and (len(ungrounded_ids) == 0)
    return {
        "결론검사_통과": is_valid,
        "engine_check_claims_전부_PASS": all_num_pass,
        "2_4쪽_외부근거_무단사용_건수": len(ungrounded_ids),
        "무단사용_상세": ungrounded_ids,
        "p1_checked_df": p1_checked_df,
    }


def create_sample_teammate_files(output_dir: str = ".") -> dict:
    """[실전 리허설용 조원 3인(산업·기업·밸류에이션) 샘플 CSV 파일 및 주장 자동 생성기]
    아직 조원들이 파일을 보내주기 전이라도, 코랩이나 내 컴퓨터에서 바로
    1) `w03_장수련_산업분석.csv` (2쪽 산업분석 15칸 표)
    2) `w03_박수연_기업분석.csv` (3쪽 기업·재무 15칸 표 — 2024·2025 DART 재무제표)
    3) `w03_조미정_밸류에이션.csv` (4쪽 밸류에이션 15칸 표 — 현재주가, 목표주가, 목표주가기대수익률)
    세 개의 실제 CSV 파일을 생성하고, 조원들이 작성한 파트별 주장(`sample_claims`)까지 세트로 돌려줍니다!
    """
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    # 1. [2쪽 산업분석 - 장수련 조원] 글로벌 HBM 출하량 성장률 및 AI 메모리 시장 규모
    df_ind_raw = pd.DataFrame([
        {
            "항목": "글로벌HBM출하량성장률",
            "값": 58.4,
            "단위": "%",
            "기준일": "2025-12-31",
            "공시일": "2026-01-20",
            "출처 URL": "https://unipass.customs.go.kr/stat/hbm2025",
        },
        {
            "항목": "글로벌AI메모리시장규모",
            "값": 42.5,
            "단위": "십억달러",
            "기준일": "2025-12-31",
            "공시일": "2026-01-20",
            "출처 URL": "https://www.wsts.org/ai-memory-2025",
        },
    ])
    rec_ind = collect_table(
        df_ind_raw,
        level=1,
        source="수출입무역통계 및 WSTS 공식 API (2쪽 산업분석 장수련)",
        reliability="상",
        basis=NOT_APPLICABLE,
        stock_code="000660",
        stock_name="SK하이닉스",
        note="2쪽 산업분석 파트 제출본",
    )
    ind_path = out / "w03_장수련_산업분석.csv"
    rec_ind.to_csv(ind_path, index=False, encoding="utf-8-sig")

    # 2. [3쪽 기업·재무분석 - 박수연 조원] 2024~2025년 DART 연결 재무제표 9줄
    df_corp_raw = pd.DataFrame([
        {"항목": "매출액(연간)", "값": 66192960000000, "단위": "원", "기준일": "2024-12-31", "공시일": "2025-03-18",
         "출처 URL": "https://dart.fss.or.kr/dsaf001/main.do?rcpNo=202503180001"},
        {"항목": "영업이익(연간)", "값": 23467319000000, "단위": "원", "기준일": "2024-12-31", "공시일": "2025-03-18",
         "출처 URL": "https://dart.fss.or.kr/dsaf001/main.do?rcpNo=202503180001"},
        {"항목": "부채총계", "값": 46200000000000, "단위": "원", "기준일": "2024-12-31", "공시일": "2025-03-18",
         "출처 URL": "https://dart.fss.or.kr/dsaf001/main.do?rcpNo=202503180001"},
        {"항목": "자본총계", "값": 73900000000000, "단위": "원", "기준일": "2024-12-31", "공시일": "2025-03-18",
         "출처 URL": "https://dart.fss.or.kr/dsaf001/main.do?rcpNo=202503180001"},
        {"항목": "매출액(연간)", "값": 97146675000000, "단위": "원", "기준일": "2025-12-31", "공시일": "2026-03-17",
         "출처 URL": "https://dart.fss.or.kr/dsaf001/main.do?rcpNo=202603170001"},
        {"항목": "영업이익(연간)", "값": 47206319000000, "단위": "원", "기준일": "2025-12-31", "공시일": "2026-03-17",
         "출처 URL": "https://dart.fss.or.kr/dsaf001/main.do?rcpNo=202603170001"},
        {"항목": "당기순이익", "값": 38500000000000, "단위": "원", "기준일": "2025-12-31", "공시일": "2026-03-17",
         "출처 URL": "https://dart.fss.or.kr/dsaf001/main.do?rcpNo=202603170001"},
        {"항목": "부채총계", "값": 55446385000000, "단위": "원", "기준일": "2025-12-31", "공시일": "2026-03-17",
         "출처 URL": "https://dart.fss.or.kr/dsaf001/main.do?rcpNo=202603170001"},
        {"항목": "자본총계", "값": 120666751000000, "단위": "원", "기준일": "2025-12-31", "공시일": "2026-03-17",
         "출처 URL": "https://dart.fss.or.kr/dsaf001/main.do?rcpNo=202603170001"},
    ])
    rec_corp = collect_table(
        df_corp_raw,
        level=1,
        source="DART 단일회사 전체 재무제표 API",
        reliability="상",
        basis="연결",
        stock_code="000660",
        stock_name="SK하이닉스",
        note="3쪽 기업·재무 파트 제출본 (박수연)",
    )
    corp_path = out / "w03_박수연_기업분석.csv"
    rec_corp.to_csv(corp_path, index=False, encoding="utf-8-sig")

    # 3. [4쪽 밸류에이션 - 조미정 조원] 현재주가, 목표주가(Base), 목표주가기대수익률, Target PBR
    df_val_raw = pd.DataFrame([
        {"항목": "현재주가(확정종가)", "값": 192000, "단위": "원", "기준일": "2025-12-31"},
        {"항목": "목표주가(Base)", "값": 240000, "단위": "원", "기준일": "2025-12-31"},
        {"항목": "목표주가기대수익률", "값": 25.0, "단위": "%", "기준일": "2025-12-31"},
        {"항목": "목표PBR(Target)", "값": 2.4, "단위": "배", "기준일": "2025-12-31"},
    ])
    rec_val = collect_table(
        df_val_raw,
        level=2,
        source="FinanceDataReader (KRX 확정종가 및 밸류에이션)",
        source_url="https://finance.naver.com/item/main.naver?code=000660",
        reliability="중",
        basis=NOT_APPLICABLE,
        stock_code="000660",
        stock_name="SK하이닉스",
        disclosed=SAME_AS_DATE,
        note="4쪽 밸류에이션 파트 제출본 (조미정)",
    )
    val_path = out / "w03_조미정_밸류에이션.csv"
    rec_val.to_csv(val_path, index=False, encoding="utf-8-sig")

    ind_src = "수출입무역통계 및 WSTS 공식 API (2쪽 산업분석 장수련)"
    dart_src = "DART 단일회사 전체 재무제표 API"
    val_src = "FinanceDataReader (KRX 확정종가 및 밸류에이션)"

    sample_claims = [
        {
            "claim_id": "C-1",
            "파트": "2쪽_산업분석(장수련)",
            "문장": "글로벌 AI 서버 수요 확대로 2025년 글로벌 HBM 출하량 성장률은 58.4%에 달하며 AI 메모리 시장 규모는 42.5십억달러를 기록했다.",
            "근거": [
                (f"000660|글로벌HBM출하량성장률|해당 없음|2025-12-31|{ind_src}", "58.4", "%"),
                (f"000660|글로벌AI메모리시장규모|해당 없음|2025-12-31|{ind_src}", "42.5", "십억달러"),
            ],
        },
        {
            "claim_id": "C-2",
            "파트": "3쪽_기업분석(박수연)",
            "문장": "SK하이닉스의 2025년 연결 매출액은 97.1조 원으로 전년 대비 46.8% 증가했고 영업이익률은 48.6%, 부채비율은 46.0%를 달성했다.",
            "근거": [
                (f"000660|매출액(연간)|연결|2025-12-31|{dart_src}", "97.1", "조 원"),
                ("M1", "46.8", "%"),
                ("M2", "48.6", "%"),
                ("M3", "46.0", "%"),
            ],
        },
        {
            "claim_id": "C-3",
            "파트": "4쪽_밸류에이션(조미정)",
            "문장": "Target PBR 2.4배를 적용해 산출한 목표주가는 240,000원으로 현재주가 192,000원 대비 기대수익률은 25.0%이다.",
            "근거": [
                (f"000660|목표PBR(Target)|해당 없음|2025-12-31|{val_src}", "2.4", "배"),
                (f"000660|목표주가(Base)|해당 없음|2025-12-31|{val_src}", "240,000", "원"),
                (f"000660|현재주가(확정종가)|해당 없음|2025-12-31|{val_src}", "192,000", "원"),
                (f"000660|목표주가기대수익률|해당 없음|2025-12-31|{val_src}", "25.0", "%"),
            ],
        },
        {
            "claim_id": "C-4",
            "파트": "3쪽_기업분석(박수연_오기테스트)",
            "문장": "(반려 테스트용 틀린 문장) 2025년 연결 영업이익률은 55.0%를 달성했다.",
            "근거": [("M2", "55.0", "%")],
        },
    ]

    return {
        "csv_files": [str(ind_path), str(corp_path), str(val_path)],
        "claims": sample_claims,
    }


def auto_detect_config_from_teammates(
    stock_code: str = None,
    year: int = None,
    target_price: float = None,
    teammate_csv_paths: list = None,
    teammate_dfs: list = None,
    teammate_claims: list = None,
    teammate_draft_texts: dict = None,
    dart_api_key: str = "",
) -> dict:
    """[센터장 자동 감지 센서]
    맨 위 설정 셀에서 `YEAR = None`, `TARGET_PRICE = None` (심지어 `STOCK_CODE = None`)으로 비워 두더라도,
    조원들이 넘겨준 CSV 표(`teammate_csv_paths`), 주장(`teammate_claims`), 또는 줄글 초안(`teammate_draft_texts`)과
    DART 최신 사업보고서 공시 연도를 스캔해서 **자동으로 종목코드·기준연도·목표주가를 찾아냅니다!**

    1) `STOCK_CODE` 자동 감지:
       - 직접 입력한 값이 없으면 조원 CSV 표의 `종목코드` 칸(거시 지표인 BOK, SP500 제외)에서 최빈값을 자동 추출합니다.
    2) `YEAR` (사업보고서 연도) 자동 감지:
       - 조원 CSV 표의 `기준일`(예: `2025-12-31`)에서 가장 최신 결산 연도를 찾거나,
       - 조원 파일이 없으면 오늘 날짜 기준으로 DART에서 가장 최근에 공시된 사업보고서 연도(예: 현재 연도 - 1)를 자동 선택합니다.
    3) `TARGET_PRICE` (4쪽 밸류에이션 목표주가) 자동 감지:
       - 1순위: 조원 CSV 표에서 항목명에 `'목표주가'` 또는 `'적정주가'`가 들어간 줄의 값을 자동으로 읽어옵니다!
       - 2순위: 조원들의 주장(`teammate_claims`)이나 4쪽 밸류에이션 줄글(`teammate_draft_texts`)에서
         `'목표주가 240,000원'`, `'적정주가 85,000원'` 패턴을 정규표현식으로 자동 추출합니다!
       - 3순위: 아직 4쪽 밸류에이션 조원이 목표주가를 안 줬다면, 임시 테스트용으로 현재주가 × 1.15(+15%)를 기본값으로 씁니다.
    """
    import re as _re

    detected_code = stock_code
    detected_year = year
    detected_target = target_price
    detection_log = []

    # 조원 CSV들을 미리 가볍게 읽어서 스캔
    loaded_dfs = []
    if teammate_csv_paths:
        for p in teammate_csv_paths:
            path_obj = Path(p)
            if path_obj.exists():
                try:
                    loaded_dfs.append(pd.read_csv(path_obj, dtype=str))
                except Exception:
                    pass
    if teammate_dfs:
        for d in teammate_dfs:
            if d is not None and not d.empty:
                loaded_dfs.append(d)

    combined_teammate_df = pd.concat(loaded_dfs, ignore_index=True) if loaded_dfs else pd.DataFrame()

    # 1. STOCK_CODE 자동 추출
    if not detected_code and not combined_teammate_df.empty and "종목코드" in combined_teammate_df.columns:
        valid_codes = [
            str(c).strip().zfill(6)
            for c in combined_teammate_df["종목코드"].dropna()
            if str(c).strip() and str(c).strip() not in ("BOK", "SP500", "해당 없음", "nan")
        ]
        if valid_codes:
            detected_code = pd.Series(valid_codes).mode().iloc[0]
            detection_log.append(f"종목코드 자동 감지(조원 CSV): {detected_code}")
    if not detected_code:
        detected_code = "000660"
        detection_log.append(f"종목코드 기본값 사용: {detected_code}")

    # 2. YEAR 자동 추출
    if not detected_year and not combined_teammate_df.empty and "기준일" in combined_teammate_df.columns:
        years_found = []
        for d_str in combined_teammate_df["기준일"].dropna():
            m = _re.match(r"^(20\d{2})", str(d_str).strip())
            if m:
                years_found.append(int(m.group(1)))
        if years_found:
            detected_year = max(years_found)
            detection_log.append(f"기준연도(YEAR) 자동 감지(조원 CSV 기준일): {detected_year}년")

    if not detected_year:
        # DART API 키가 있으면 최근 연도부터 사업보고서가 있는지 자동 탐색, 없으면 전년도(예: 2025년)
        cur_y = datetime.now(KST).year
        detected_year = cur_y - 1
        if dart_api_key:
            for candidate_y in (cur_y - 1, cur_y - 2):
                try:
                    test_df = collect_dart(detected_code, candidate_y, 4, "CFS", dart_api_key)
                    if test_df is not None and not test_df.empty:
                        detected_year = candidate_y
                        break
                except Exception:
                    continue
        detection_log.append(f"기준연도(YEAR) 자동 설정(최신 사업보고서 연도): {detected_year}년")

    # 3. TARGET_PRICE (4쪽 밸류에이션 목표주가) 자동 추출
    if not detected_target and not combined_teammate_df.empty and "항목" in combined_teammate_df.columns:
        tp_rows = combined_teammate_df[
            combined_teammate_df["항목"].astype(str).str.contains("목표주가|적정주가", case=False, na=False)
            & ~combined_teammate_df["항목"].astype(str).str.contains("수익률|괴리율|PBR|PER", case=False, na=False)
        ]
        if not tp_rows.empty and "값" in tp_rows.columns:
            num_val, _ = to_number(tp_rows.iloc[-1]["값"])
            if num_val is not None and num_val > 0:
                detected_target = float(num_val)
                detection_log.append(f"목표주가(TARGET_PRICE) 자동 감지(4쪽 밸류에이션 조원 표): {int(detected_target):,}원")

    # 조원 주장(teammate_claims)이나 초안 줄글(teammate_draft_texts)에서 "목표주가 OOO원" 정규식 스캔
    if not detected_target:
        texts_to_scan = []
        if teammate_claims:
            for c in teammate_claims:
                texts_to_scan.append(str(c.get("문장", "")))
        if teammate_draft_texts:
            for t in teammate_draft_texts.values():
                texts_to_scan.append(str(t))
        full_scan_text = " ".join(texts_to_scan)
        m_tp = _re.search(r"(?:목표주가|적정주가)[^\d]{0,15}([\d,]+)\s*원", full_scan_text)
        if m_tp:
            num_val, _ = to_number(m_tp.group(1))
            if num_val is not None and num_val > 0:
                detected_target = float(num_val)
                detection_log.append(f"목표주가(TARGET_PRICE) 자동 감지(조원 밸류에이션 문장): {int(detected_target):,}원")

    # 아직 조원이 목표주가를 안 줬을 때의 안전한 자동 계산
    if not detected_target:
        if detected_code == "000660":
            detected_target = 240000.0
            detection_log.append("목표주가(TARGET_PRICE) 기본 예시값 사용(4쪽 미입력 시): 240,000원")
        else:
            # 실시간 현재주가를 조회해 임시로 +15% 수준을 기본 목표주가로 세팅
            cur_p = 100000.0
            try:
                import FinanceDataReader as fdr
                start = (datetime.now(KST) - timedelta(days=20)).strftime("%Y-%m-%d")
                df_p = fdr.DataReader(detected_code, start)
                if not df_p.empty:
                    cur_p = float(df_p["Close"].iloc[-1])
            except Exception:
                pass
            detected_target = round(cur_p * 1.15, -2)
            detection_log.append(
                f"목표주가(TARGET_PRICE) 임시 자동 산출(4쪽 조원 파일 대기 중 -> 현재가 {int(cur_p):,}원 +15%): {int(detected_target):,}원"
            )

    return {
        "STOCK_CODE": detected_code,
        "YEAR": int(detected_year),
        "TARGET_PRICE": float(detected_target),
        "감지_로그": detection_log,
    }


def merge_teammates_tables_and_metrics(
    base_records: pd.DataFrame,
    stock_code: str,
    stock_name: str,
    year: int,
    teammate_csv_paths: list = None,
    teammate_dfs: list = None,
    teammate_metric_specs: list = None,
) -> tuple:
    """[출구 ④ 조원 데이터·지표 통합 허브]
    조원들(2쪽 산업분석 장수련, 3쪽 기업·재무 박수연, 4쪽 밸류에이션 조미정)이
    수집한 15칸 표(CSV 파일 또는 DataFrame)와 조원들이 설계한 추가 지표 정의(METRIC_SPECS)를 받아와서
    박형민님의 원본 엔진(`with_ids` -> `build_sources` -> `compute_metrics`)으로 한 번에 통합합니다!

    - 출처 번호(`S#`)를 전체 합산 기준으로 딱 한 번 다시 매겨서 충돌(T10 등)을 방지합니다.
    - 같은 사실(`rid`)이 조원들 사이에 중복되면 `deduplicate` 옵션으로 안전하게 병합합니다.
    """
    frames = [base_records] if base_records is not None and not base_records.empty else []

    if teammate_csv_paths:
        for p in teammate_csv_paths:
            path_obj = Path(p)
            if path_obj.exists():
                df_csv = pd.read_csv(path_obj, dtype=str)
                keep_cols = [c for c in COLUMNS if c in df_csv.columns]
                df_clean = df_csv[keep_cols].copy()
                if "값" in df_clean.columns:
                    df_clean["값"] = df_clean["값"].map(lambda v: to_number(v)[0])
                frames.append(df_clean)
                print(f"📥 조원 CSV 로드 완료: {path_obj.name} ({len(df_clean)}줄)")
            else:
                print(f"⏭ 조원 CSV 파일 없음(건너뜀): {p}")

    if teammate_dfs:
        for idx, t_df in enumerate(teammate_dfs, 1):
            if t_df is not None and not t_df.empty:
                keep_cols = [c for c in COLUMNS if c in t_df.columns]
                df_clean = t_df[keep_cols].copy()
                if "값" in df_clean.columns:
                    df_clean["값"] = df_clean["값"].map(lambda v: to_number(v)[0])
                frames.append(df_clean)
                print(f"📥 조원 데이터프레임 #{idx} 병합 완료 ({len(df_clean)}줄)")

    if not frames:
        raise CollectError("병합할 데이터 표가 하나도 없어요.")

    merged_raw = pd.concat(frames, ignore_index=True)

    # 조원들이 같은 DART 재무제표 줄을 중복해서 가져왔을 수 있으므로 (종목코드, 항목, 기준, 기준일, 출처) 중복 제거
    dedup_cols = ["종목코드", "항목", "기준", "기준일", "출처"]
    before_len = len(merged_raw)
    merged_raw = merged_raw.drop_duplicates(subset=dedup_cols, keep="first").reset_index(drop=True)
    if len(merged_raw) < before_len:
        print(f"🧹 조원 간 중복된 동일 사실(rid) {before_len - len(merged_raw)}줄 자동 정리 완료")

    # 출구 ④ 약속: 합친 표 전체에 대해 with_ids()와 build_sources()를 다시 돌려 S# 번호를 통일
    sources_df, tagged_records = build_sources(with_ids(merged_raw))

    # 기본 M1~M3 지표에 조원들이 추가한 지표(예: M4, M5...)를 합쳐서 계산
    all_specs = list(METRIC_SPECS)
    if teammate_metric_specs:
        existing_ids = {s["metric_id"] for s in all_specs}
        for spec in teammate_metric_specs:
            if spec["metric_id"] not in existing_ids:
                all_specs.append(spec)
                existing_ids.add(spec["metric_id"])

    base_date = f"{year}-12-31"
    metrics_df = compute_metrics(tagged_records, all_specs, stock_code, base_date)
    return sources_df, tagged_records, metrics_df


def synthesize_teammates_to_page1(
    stock_code: str,
    stock_name: str,
    year: int,
    tagged_records: pd.DataFrame,
    metrics_df: pd.DataFrame,
    sources_df: pd.DataFrame,
    rating_decision: dict,
    teammate_claims: list = None,
    teammate_draft_texts: dict = None,
    gemini_api_key: str = "",
) -> dict:
    """[출구 ④·⑤ 센터장 메인 오케스트레이터]
    조원들이 넘겨준 (A) 구조화된 주장 리스트(`teammate_claims`) 또는 (B) 파트별 초안 텍스트(`teammate_draft_texts`)를 받아서:
    1) `check_claims()`로 숫자 대조를 수행하고,
    2) `PASS`한 조원들의 주장만 골라내며(`FIX`·`REVIEW`는 해당 파트로 반려 목록 생성),
    3) 통과한 조원들의 핵심 근거(`rid`, `M#`)와 센터장의 동적 밴드 투자의견(`BUY/HOLD/SELL`)을 결합해
       **1쪽 투자 요약 주장(`page1_claims`)**과 **검증 기록지**를 자동으로 생성합니다!
    """
    collected_claims = []

    # 1. 조원들이 이미 [{"claim_id": ..., "파트": ..., "문장": ..., "근거": [...]}] 형태로 넘겨준 경우
    if teammate_claims:
        collected_claims.extend(teammate_claims)

    # 2. 조원들이 줄글 초안(예: {"2쪽_산업분석": "...", "3쪽_기업재무": "..."})으로 넘겨준 경우 Gemini로 근거 자동 추출
    if teammate_draft_texts and gemini_api_key:
        for part_name, draft_text in teammate_draft_texts.items():
            if not draft_text or not draft_text.strip():
                continue
            prompt = claims_prompt(tagged_records, metrics_df, draft_text)
            tmp_path = Path(f"tmp_claims_{stock_code}_{part_name}.json")
            try:
                ask_claims(
                    prompt, gemini_api_key, tmp_path,
                    model="gemini-2.5-flash", fallback_model="gemini-2.0-flash"
                )
                extracted = load_claims(tmp_path)
                for idx, c in enumerate(extracted, 1):
                    c["claim_id"] = f"{part_name[:2]}-{idx}"
                    c["파트"] = part_name
                    collected_claims.append(c)
            except Exception as e:
                print(f"⚠ [{part_name}] Gemini 주장 추출 건너뜀: {e}")

    # 3. 만약 조원 주장이 아직 없으면 현재 표(tagged_records, metrics_df)에서 기본 주장을 자동 생성
    if not collected_claims:
        collected_claims, _ = build_dynamic_claims_for_any_stock(
            stock_code=stock_code,
            stock_name=stock_name,
            year=year,
            tagged_records=tagged_records,
            metrics_df=metrics_df,
            rating_decision=rating_decision,
        )

    # 4. 주장 검증 (check_claims) 및 PASS / 반려(FIX, REVIEW) 분류
    checked_df = check_claims(collected_claims, metrics_df, tagged_records)
    passed_claims, rejected_claims = filter_passed_claims(collected_claims, checked_df)

    # 5. 통과한 조원들의 주장(passed_claims)을 바탕으로 1쪽 투자포인트(page1_claims) 조립
    page1_claims = []
    for idx, pc in enumerate(passed_claims, 1):
        part_label = pc.get("파트", f"파트{idx}")
        # 밸류에이션 파트 주장이거나 마지막 주장이면 투자의견 밴드 소제목 부여
        if "밸류" in part_label or "4쪽" in part_label:
            title = (
                f"[{part_label} · 투자의견 {rating_decision['투자의견']}] "
                f"기대수익률({rating_decision['기대수익률_pct']:+.2f}%) vs "
                f"보유 밴드(한은 금리 {rating_decision['하단기준_한국은행기준금리_Rf_pct']:.2f}% ~ "
                f"S&P500 10년 평균 {rating_decision['상단기준_SP500_10년평균_pct']:.2f}%)"
            )
        else:
            title = f"[{part_label} 핵심 요약] {pc['문장'][:42]}..."

        page1_claims.append({
            "claim_id": f"P1-{idx}",
            "원본_claim_id": pc["claim_id"],
            "담당_파트": part_label,
            "소제목": title,
            "문장": pc["문장"],
            "근거": list(pc.get("근거", [])),
        })

    # 6. 출구 ⑤ 1쪽 결론 무결성 검사 (2~4쪽 PASS 근거만 썼는지 + 숫자 대조 100% PASS인지)
    conclusion_check = verify_page1_conclusion(
        page1_claims=page1_claims,
        passed_part_claims=passed_claims,
        metrics_df=metrics_df,
        records_df=tagged_records,
    )

    return {
        "checked_df": checked_df,
        "passed_claims": passed_claims,
        "rejected_claims": rejected_claims,
        "page1_claims": page1_claims,
        "conclusion_check": conclusion_check,
    }


# ==============================================================================
# [SMIC 서울대 투자연구회 농심홀딩스(072710) 보고서 포맷] 센터장 1쪽 & KEY CHARTS 프롬프트 및 렌더러
# ==============================================================================

CHIEF_SMIC_SYSTEM_PROMPT = """# [역할 정의 (Role)]
당신은 KFA AI 리서치 센터 1조의 **센터장(Chief Equity Research Director, 김규민)** 에이전트입니다.
당신의 임무는 조원들(2쪽 산업분석 장수련, 3쪽 기업·재무분석 박수연, 4쪽 밸류에이션 조미정)이 수집한 15칸 데이터 표(`tagged_records`), 파생 지표(`metrics_df`: M1~M3), 그리고 원본 대조(`check_claims()`)에서 **`PASS` 판정을 받은 주장(`passed_claims`)만**을 통합(출구 ④)하여, **서울대 투자연구회(SMIC) 리서치 보고서 표지(Page 1) 및 핵심 차트(Page 2: KEY CHARTS) 양식**과 동일한 고품격 1쪽 투자 요약 보고서를 작성하고 최종 투자의견을 판정(출구 ⑤)하는 것입니다.

---

# [절대 준수 원칙: KFA 엔진 무결성 5대 규칙 (Guardrails)]
1. **임의 수치 창작 절대 금지 (Zero Hallucination)**:
   - 제공된 `tagged_records`(원본 사실 표), `metrics_df`(계산된 지표 표), `rating_decision`(동적 밴드 판정 결과), 그리고 `PASS`한 `passed_claims`에 없는 숫자는 **단 하나도 본문에 쓰지 마십시오.**
2. **반려(`FIX` / `REVIEW`) 주장 수록 엄금**:
   - 조원의 주장 중 `FIX`(수치 불일치) 또는 `REVIEW`(근거 누락) 판정을 받은 주장은 1쪽 표지 본문에 절대 반영하지 말고, 하단 **[검증 기록지 요약 (반려 및 수정 요청 내역)]** 섹션에만 별도로 명시하십시오.
3. **모든 문장 끝 출처 번호(`[S#]`) 자동 표기**:
   - 각 문장에 사용된 근거(`rid` 또는 `M#`)에 연결된 통합 출처 번호(예: `[S1]`, `[S1, S4]`)를 문장이나 단락 끝에 반드시 표기하십시오.
4. **센터장 고유 동적 벤치마크 밴드(`Rf ~ S&P 500 10Y CAGR`) 투자의견 판정 규칙**:
   - $\\text{기대수익률(\\%)} = \\frac{\\text{목표주가(4쪽 밸류)} - \\text{현재주가}}{\\text{현재주가}} \\times 100$
   - **🟢 `BUY (매수)`**: $\\text{기대수익률} \\ge \\text{S\\&P 500 10년 연평균 수익률(CAGR)}$ *(글로벌 대표 지수 10년 장기 복리 수익률을 상회하는 초과수익 기대)*
   - **🟡 `HOLD (보유)`**: $\\text{현 한국은행 기준금리(무위험이자율 } R_f) \\le \\text{기대수익률} < \\text{S\\&P 500 10년 연평균 수익률}$ *(무위험자산 수익률은 상회하나 글로벌 기회비용 대비 중립)*
   - **🔴 `SELL (매도)`**: $\\text{기대수익률} < \\text{현 한국은행 기준금리(무위험이자율 } R_f)$ *(무위험이자율조차 하회하므로 투자 매력 부재)*
5. **`check_claims()` 숫자·단위 완벽 일치**:
   - 본문에 적는 숫자와 단위는 원본 표(`원`, `조 원`, `%`, `배`) 및 `PASS` 문장의 표기와 정확히 일치시켜 재검증 시 100% `PASS`가 유지되도록 하십시오.

---

# [출력 형식: SMIC (서울대 투자연구회) 농심홀딩스 보고서 1~2쪽 완벽 재현 레이아웃]
반드시 아래의 **SMIC 리서치 커버 페이지 구조(상단 헤더 배너 + 좌측 메인 내러티브 & 하단 요약 손익계산서 + 우측 사이드바 투자지표 박스 + 2쪽 KEY CHARTS)** 순서와 형식을 지켜 출력하십시오.

## 1. [상단 헤더 배너 (Header Banner)]
- **발행 조직 및 일자**: `KFA AI 리서치 센터 1조 | YYYY년 M월 D일`
- **종목명 및 종목코드**: `# {종목명} ({종목코드})`
- **메인 캐치프레이즈 (One-Line Headline)**: SMIC 보고서의 *"밸류업과 함께 라면"*처럼, 해당 기업의 핵심 산업 모멘텀과 리레이팅 논리를 관통하는 강렬한 한 줄 헤드라인을 제시하십시오.

## 2. [우측 사이드바 요약 박스 (Right Sidebar: Rating & Key Data Box)]
- **투자의견 및 목표주가 박스**:
  - `Rating`: **`BUY` / `HOLD` / `SELL`**
  - `현재주가`: `OOO,OOO 원`
  - `목표주가`: `OOO,OOO 원`
  - `상승여력 (기대수익률)`: `+OO.O%`
  - `동적 보유(HOLD) 밴드`: `현 한은 기준금리(Rf) O.OO% ~ S&P 500 10Y CAGR OO.OO%`
- **Key Metrics (핵심 투자지표 요약)**:
  - `매출성장률 (M1)` / `영업이익률 (M2)` / `목표 멀티플(PBR/PER)` / `S&P 500 대비 초과수익률(%p)`
- **B/S Data (재무상태표 스냅샷, 기준연도)**:
  - `자산 총계 (= 부채총계 + 자본총계)` / `부채 총계 (부채비율 M3)` / `자본 총계`
- **Research Team (KFA 1조 리서치 센터)**:
  - `센터장`: 김규민 (1쪽 투자요약 · 출구 ④·⑤ 결론)
  - `산업분석(2쪽)`: 장수련 | `기업·재무(3쪽)`: 박수연 | `밸류에이션(4쪽)`: 조미정 | `검증관`: 채주성 | `공통엔진`: 박형민

## 3. [좌측 메인 본문 (Main Narrative Column)]
- **[Executive Hook: 도입부 소제목]** (예: *"AI 메모리 슈퍼사이클이 선도 기업을 보라고 한다"*)
  - 왜 지금 시장이 이 산업과 이 종목을 주목해야 하는지 거시·산업적 배경과 핵심 결론을 3~4문장으로 압축 제시합니다.
- **[투자포인트 1: 캐치프레이즈형 제목]** (2쪽 산업분석 장수련 `PASS` 주장 합성)
  - 전방 산업의 구조적 성장률, 시장 규모 및 동사의 시장 지배력 근거(`rid`/`M#`)를 인용하여 1개 단락(3~5문장)으로 서술하고 끝에 `[S#]`를 표기합니다.
- **[투자포인트 2: 캐치프레이즈형 제목]** (3쪽 기업·재무분석 박수연 `PASS` 주장 합성)
  - 전년도 대비 최근 결산연도의 매출액 성장(`M1`), 영업이익 및 영업이익률(`M2`), 재무건전성(`부채비율 M3`) 개선 폭을 구체적 숫자와 함께 1개 단락(3~5문장)으로 서술하고 끝에 `[S#]`를 표기합니다.
- **[Valuation: 목표주가 산출 근거 및 동적 밴드 대조]** (4쪽 밸류에이션 조미정 `PASS` 주장 + 센터장 동적 밴드 판정)
  - 4쪽 밸류에이션 파트의 목표 멀티플 및 적정주가 산출 논리를 제시하고, 현 주가 대비 기대수익률이 **한국은행 기준금리($R_f$) ~ S&P 500 10년 연평균 수익률(CAGR)** 동적 밴드에서 어디에 위치하는지 대조하여 최종 **`BUY / HOLD / SELL`** 투자의견을 선언합니다 (`[S#]` 표기).

## 4. [하단 요약 재무제표 (추정 연결포괄손익계산서 및 재무비율 요약 Table)]
- SMIC 보고서 1쪽 하단 표와 동일하게 `(YEAR-1)년` vs `(YEAR)년`의 **`매출액`, `YoY(매출성장률 M1)`, `영업이익`, `OPM(영업이익률 M2)`, `당기순이익`, `NPM(순이익률)`, `부채총계`, `자본총계`, `부채비율(M3)`** 비교표를 제시합니다.

## 5. [Page 2: KEY CHARTS & 검증 기록지 (출구 ④·⑤ 감사 로그)]
- **KEY CHART 1**: 투자포인트 & 동적 벤치마크 밴드(`Rf ~ S&P 500`) 의사결정 구조도
- **KEY CHART 2**: 연도별 실적 성장 및 수익성 지표 요약 차트/표 (`출처: ... [S#]`)
- **검증 기록지 요약**: 조원 주장 전수 검사(`PASS` 수록 건수 vs `FIX/REVIEW` 반려 건수) 및 출구 ⑤ 1쪽 결론 무결성 점검 결과
"""


def build_smic_prompt_for_stock(
    stock_code: str,
    stock_name: str,
    year: int,
    tagged_records: pd.DataFrame,
    metrics_df: pd.DataFrame,
    sources_df: pd.DataFrame,
    rating_decision: dict,
    page1_claims: list,
     conclusion_check: dict,
    rejected_claims: list = None,
) -> str:
    """현재 종목의 통합 데이터(`tagged_records`, `metrics_df`, `sources_df`, `rating_decision`, `page1_claims`)를
    SMIC 농심홀딩스 보고서 양식 프롬프트(`CHIEF_SMIC_SYSTEM_PROMPT`)에 자동으로 채워 넣어
    LLM(Gemini / ChatGPT / Claude)에 바로 입력하거나 코랩에서 호출할 수 있는 완성형 프롬프트 문자열을 반환합니다.
    """
    p1_checked_df = conclusion_check.get("p1_checked_df", pd.DataFrame())
    claims_text_lines = []
    for p1c in page1_claims:
        cid = p1c["claim_id"]
        s_label = ""
        if not p1_checked_df.empty and cid in p1_checked_df["claim_id"].values:
            s_label = str(p1_checked_df[p1_checked_df["claim_id"] == cid]["출처 번호"].iloc[0])
        claims_text_lines.append(
            f"- [{cid} | 담당: {p1c.get('담당_파트', '')}] 문장: \"{p1c['문장']}\" | 근거ID: {p1c.get('근거', [])} | 출처번호: {s_label}"
        )

    rejected_lines = []
    if rejected_claims:
        for rc in rejected_claims:
            rejected_lines.append(
                f"- [반려 {rc.get('claim_id', '')} | {rc.get('파트', '')}] 판정: {rc.get('판정_목록', '')} | 사유: {rc.get('탈락_이유', '')}"
            )
    else:
        rejected_lines.append("- 반려된 주장 없음 (조원 제출 주장 전원 PASS)")

    records_cols = [c for c in ["rid", "종목코드", "종목명", "항목", "값", "단위", "기준", "기준일", "출처", "sid"] if c in tagged_records.columns]
    metrics_cols = [c for c in ["metric_id", "지표명", "값", "단위", "판정", "근거_rid"] if c in metrics_df.columns]

    user_context = f"""
---
# [입력 데이터 (Input Context — 오직 아래의 검증된 숫자만 사용할 것)]

## 1) 분석 대상 종목 및 기준연도
- 종목명(종목코드): {stock_name} ({stock_code})
- 기준연도(YEAR): {year}년 (비교연도: {year - 1}년)
- 작성일: {datetime.now(KST).strftime('%Y년 %m월 %d일')}

## 2) 센터장 동적 벤치마크 밴드 판정 결과 (`rating_decision`)
- 현재주가: {int(rating_decision['현재주가']):,}원
- 목표주가(4쪽 밸류에이션): {int(rating_decision['목표주가']):,}원
- 목표주가 기대수익률(상승여력): {rating_decision['기대수익률_pct']:+.2f}%
- 하단 기준 (현 한국은행 기준금리, 무위험이자율 Rf): {rating_decision['하단기준_한국은행기준금리_Rf_pct']:.2f}%
- 상단 기준 (S&P 500 최근 10년 연평균 수익률 CAGR): {rating_decision['상단기준_SP500_10년평균_pct']:.2f}%
- 동적 보유(HOLD) 밴드 구간: {rating_decision['보유_HOLD_구간']}
- 최종 투자의견(Rating): **{rating_decision['투자의견']}**
- 판정 사유: {rating_decision['판정_근거']}

## 3) 조원(2~4쪽) 검증 통과(`PASS`) 핵심 주장 목록 (`page1_claims`)
{chr(10).join(claims_text_lines)}

## 4) 통합 계산 지표 표 (`metrics_df`: M1~M3)
{metrics_df[metrics_cols].to_string(index=False)}

## 5) 통합 원본 사실 표 (`tagged_records`: rid & sid 매핑)
{tagged_records[records_cols].to_string(index=False)}

## 6) 통합 출처 목록 (`sources_df`: S# 매핑)
{sources_df.to_string(index=False)}

## 7) 검증 기록지 반려 목록 (`rejected_claims` — 본문 수록 금지, 하단 기록지에만 기재)
{chr(10).join(rejected_lines)}

---
# [최종 지시사항]
위 **[입력 데이터]**만을 사용하여, **[출력 형식: SMIC 농심홀딩스 보고서 1~2쪽 완벽 재현 레이아웃]**에 맞춰 `{stock_name} ({stock_code})`의 1쪽 커버 리포트와 2쪽 KEY CHARTS를 작성하십시오.
"""
    return CHIEF_SMIC_SYSTEM_PROMPT.strip() + "\n" + user_context.strip()


def render_smic_style_page1_report(
    stock_code: str,
    stock_name: str,
    year: int,
    tagged_records: pd.DataFrame,
    metrics_df: pd.DataFrame,
    sources_df: pd.DataFrame,
    rating_decision: dict,
    page1_claims: list,
    conclusion_check: dict,
    rejected_claims: list = None,
) -> dict:
    """SMIC(서울대 투자연구회) '농심홀딩스(072710) - 밸류업과 함께 라면' 보고서의 1쪽(2단 레이아웃:
    상단 네이비 배너 + 좌측 본문[도입부 헤드라인 / 투자포인트 1 / 투자포인트 2 / Valuation / 하단 손익계산서 요약표]
    + 우측 사이드바[Rating / 현재주가 / 목표주가 / 상승여력 / Key Metrics / B/S Data / 리서치팀]) 및
    2쪽(KEY CHARTS) 디자인을 **HTML(코랩 2단 시각 렌더링용)**과 **Markdown(파일 저장용)**으로 동시에 생성합니다!
    모든 본문 문장은 `check_claims()` 100% `PASS`한 문장과 정확한 수치만 사용합니다.
    """
    today_str = datetime.now(KST).strftime("%Y년 %m월 %d일")
    p1_checked_df = conclusion_check.get("p1_checked_df", pd.DataFrame())

    def _get_val(code_val: str, item_name: str, y_val: int = None):
        sub = tagged_records[
            (tagged_records["종목코드"] == code_val)
            & (tagged_records["항목"] == item_name)
        ]
        if y_val is not None:
            sub = sub[sub["기준일"].astype(str).str.startswith(str(y_val))]
        if sub.empty:
            return None, None
        row = sub.iloc[-1]
        return float(row["값"]), str(row.get("sid", ""))

    def _get_metric(mid: str):
        sub = metrics_df[metrics_df["metric_id"] == mid]
        if sub.empty or pd.isna(sub.iloc[0]["값"]):
            return None
        return float(sub.iloc[0]["값"])

    rev_prev, s_rev_prev = _get_val(stock_code, "매출액(연간)", year - 1)
    rev_curr, s_rev_curr = _get_val(stock_code, "매출액(연간)", year)
    op_prev, _ = _get_val(stock_code, "영업이익(연간)", year - 1)
    op_curr, s_op_curr = _get_val(stock_code, "영업이익(연간)", year)
    ni_prev, _ = _get_val(stock_code, "당기순이익", year - 1)
    ni_curr, _ = _get_val(stock_code, "당기순이익", year)
    debt_prev, _ = _get_val(stock_code, "부채총계", year - 1)
    debt_curr, _ = _get_val(stock_code, "부채총계", year)
    eq_prev, _ = _get_val(stock_code, "자본총계", year - 1)
    eq_curr, _ = _get_val(stock_code, "자본총계", year)

    m1_growth = _get_metric("M1")
    m2_opm = _get_metric("M2")
    m3_debt_ratio = _get_metric("M3")

    opm_prev = (op_prev / rev_prev * 100.0) if (op_prev is not None and rev_prev) else None
    npm_prev = (ni_prev / rev_prev * 100.0) if (ni_prev is not None and rev_prev) else None
    npm_curr = (ni_curr / rev_curr * 100.0) if (ni_curr is not None and rev_curr) else None
    debt_ratio_prev = (debt_prev / eq_prev * 100.0) if (debt_prev is not None and eq_prev) else None

    asset_curr = (debt_curr + eq_curr) if (debt_curr is not None and eq_curr is not None) else None

    def _fmt_trillion(v):
        if v is None:
            return "-"
        tril = v / 1e12
        if abs(tril) >= 1.0:
            return f"{tril:,.2f}조 원"
        eok = v / 1e8
        return f"{eok:,.0f}억 원"

    def _fmt_eok(v):
        if v is None:
            return "-"
        return f"{v / 1e8:,.0f}"

    def _fmt_pct(v, sign=False):
        if v is None:
            return "-"
        return f"{v:+.2f}%" if sign else f"{v:.2f}%"

    # 파트별 통과 주장 분리 (2쪽 산업, 3쪽 기업·재무, 4쪽 밸류에이션)
    ind_claims = []
    corp_claims = []
    val_claims = []
    for p1c in page1_claims:
        cid = p1c["claim_id"]
        s_label = ""
        if not p1_checked_df.empty and cid in p1_checked_df["claim_id"].values:
            s_label = str(p1_checked_df[p1_checked_df["claim_id"] == cid]["출처 번호"].iloc[0])
        entry = {**p1c, "출처_라벨": s_label}
        part_str = str(p1c.get("담당_파트", ""))
        if "산업" in part_str or "2쪽" in part_str:
            ind_claims.append(entry)
        elif "기업" in part_str or "재무" in part_str or "3쪽" in part_str:
            corp_claims.append(entry)
        elif "밸류" in part_str or "4쪽" in part_str:
            val_claims.append(entry)
        else:
            corp_claims.append(entry)

    # 만약 특정 파트 주장이 비어 있으면 전체 page1_claims에서 순서대로 배분
    if not ind_claims and page1_claims:
        ind_claims = [{**page1_claims[0], "출처_라벨": "[S1]"}]
    if not corp_claims and len(page1_claims) >= 2:
        corp_claims = [{**page1_claims[1], "출처_라벨": "[S1]"}]
    if not val_claims and page1_claims:
        val_claims = [{**page1_claims[-1], "출처_라벨": "[S2]"}]

    rating_str = rating_decision["투자의견"]
    upside_pct = rating_decision["기대수익률_pct"]
    rf_pct = rating_decision["하단기준_한국은행기준금리_Rf_pct"]
    sp500_pct = rating_decision["상단기준_SP500_10년평균_pct"]
    cur_price = int(rating_decision["현재주가"])
    tgt_price = int(rating_decision["목표주가"])
    alpha_vs_sp500 = upside_pct - sp500_pct

    # 목표 멀티플(목표PBR 등) 탐색
    target_pbr_val, _ = _get_val(stock_code, "목표PBR(Target)")
    target_mult_str = f"Target PBR {target_pbr_val:.2f}배" if target_pbr_val else "목표주가 산출 모델"

    # SMIC 스타일 헤드라인 & 소제목 자동 구성
    if stock_code == "000660":
        main_catchphrase = "AI 메모리 슈퍼사이클과 함께 달리는 HBM 절대 강자"
        hook_title = "글로벌 AI 인프라 투자가 HBM 선도 기업을 보라고 한다"
        tp1_title = "투자포인트 1: HBM 초격차(超格差), AI 메모리 시장을 지배하다 (2쪽 산업분석 종합)"
        tp2_title = f"투자포인트 2: 숫자로 증명된 체질 개선, 영업이익률 {_fmt_pct(m2_opm)}의 레버리지 (3쪽 기업·재무 종합)"
    else:
        main_catchphrase = f"{year}년 구조적 실적 턴어라운드와 리레이팅의 원년"
        hook_title = f"펀더멘털 개선과 동적 벤치마크 밴드가 {stock_name}을(를) 가리킨다"
        tp1_title = "투자포인트 1: 전방 산업 모멘텀과 핵심 시장 지배력 확대 (2쪽 산업분석 종합)"
        tp2_title = f"투자포인트 2: 외형 성장({_fmt_pct(m1_growth, sign=True)})과 수익성({_fmt_pct(m2_opm)})의 동반 도약 (3쪽 기업·재무 종합)"

    hook_body = (
        f"글로벌 거시 환경의 불확실성 속에서도 구조적 성장 산업 내 독점적 지위를 확보한 선도 기업으로의 이익 쏠림 현상이 심화되고 있다. "
        f"{stock_name}({stock_code})은(는) {year}년 연간 매출액 {_fmt_trillion(rev_curr)}(YoY {_fmt_pct(m1_growth, sign=True)}), "
        f"영업이익 {_fmt_trillion(op_curr)}(영업이익률 {_fmt_pct(m2_opm)})을 기록하며 펀더멘털의 구조적 레벨업을 입증하였다. "
        f"현 한국은행 기준금리({rf_pct:.2f}%)와 S&P 500 최근 10년 연평균 수익률({sp500_pct:.2f}%)을 양 축으로 하는 "
        f"1조 리서치 센터의 동적 벤치마크 모델 대조 결과, 동사의 기대수익률은 {upside_pct:+.2f}%로 산출되어 최종 투자의견 '{rating_str}'을 제시한다."
    )

    tp1_body = " ".join([f"{c['문장']} {c['출처_라벨']}" for c in ind_claims])
    tp2_body = " ".join([f"{c['문장']} {c['출처_라벨']}" for c in corp_claims])
    val_body = (
        " ".join([f"{c['문장']} {c['출처_라벨']}" for c in val_claims])
        + f" 아울러 센터장 동적 벤치마크 판정 기준(보유 구간: 한은 기준금리 {rf_pct:.2f}% ~ S&P 500 10Y CAGR {sp500_pct:.2f}%)과 대조할 때, "
        f"동사의 목표주가 기대수익률({upside_pct:+.2f}%)은 S&P 500 10년 평균 대비 {alpha_vs_sp500:+.2f}%p 수준이므로 최종 투자의견 '{rating_str}'으로 결론짓는다."
    )

    # Rating 색상 테마
    if "BUY" in rating_str:
        badge_bg = "#1b365d"
        badge_accent = "#d9381e"
    elif "HOLD" in rating_str:
        badge_bg = "#1b365d"
        badge_accent = "#d97706"
    else:
        badge_bg = "#1b365d"
        badge_accent = "#2563eb"

    # ------------------------------------------------------------------
    # 1) 마크다운(Markdown) 버전 생성 (w05_page1_종목코드_연도.md 저장용)
    # ------------------------------------------------------------------
    md_lines = [
        f"# 🏛️ [KFA AI 리서치 센터 1조 | SMIC 포맷] {stock_name} ({stock_code})",
        f"### **\"{main_catchphrase}\"**",
        f"> **발행일**: {today_str} | **센터장(1쪽·출구④⑤)**: 김규민 | **산업(2쪽)**: 장수련 | **기업·재무(3쪽)**: 박수연 | **밸류에이션(4쪽)**: 조미정 | **검증관**: 채주성 | **공통엔진**: 박형민",
        "",
        "---",
        "",
        "## 📌 [Right Sidebar] 투자의견 & 핵심 지표 스냅샷 (SMIC 우측 사이드바 양식)",
        "| 구분 | 핵심 수치 / 내용 | 구분 | 핵심 수치 / 내용 |",
        "| :--- | :--- | :--- | :--- |",
        f"| **투자의견 (Rating)** | **{rating_str}** | **매출성장률 (M1, YoY)** | **{_fmt_pct(m1_growth, sign=True)}** |",
        f"| **현재주가** | **{cur_price:,} 원** | **영업이익률 (M2, OPM)** | **{_fmt_pct(m2_opm)}** |",
        f"| **목표주가 (Target Price)** | **{tgt_price:,} 원** ({target_mult_str}) | **S&P 500 대비 초과수익률** | **{alpha_vs_sp500:+.2f}%p** |",
        f"| **상승여력 (Upside)** | **{upside_pct:+.2f}%** | **자산 총계 ({str(year)[2:]}년)** | **{_fmt_trillion(asset_curr)}** |",
        f"| **하단: 한은 기준금리 ($R_f$)** | **{rf_pct:.2f}%** | **부채 총계 (부채비율 M3)** | **{_fmt_trillion(debt_curr)} ({_fmt_pct(m3_debt_ratio)})** |",
        f"| **상단: S&P 500 10Y CAGR** | **{sp500_pct:.2f}%** | **자본 총계 ({str(year)[2:]}년)** | **{_fmt_trillion(eq_curr)}** |",
        "",
        "---",
        "",
        f"## 📰 [Main Column] {hook_title}",
        f"{hook_body}",
        "",
        f"### ■ {tp1_title}",
        f"{tp1_body}",
        "",
        f"### ■ {tp2_title}",
        f"{tp2_body}",
        "",
        f"### ■ Valuation: 목표주가 {tgt_price:,}원, 투자의견 {rating_str} 제시",
        f"{val_body}",
        "",
        "---",
        "",
        f"## 📊 [Summary Table] 연결포괄손익계산서 및 재무비율 요약 (단위: 억 원, %)",
        f"| 항목 (연결 기준) | {year - 1}년 (실적) | {year}년 (실적) | 증감 / 핵심 지표 (M1~M3) |",
        "| :--- | :---: | :---: | :---: |",
        f"| **매출액 (억 원)** | {_fmt_eok(rev_prev)} | **{_fmt_eok(rev_curr)}** | **YoY (M1): {_fmt_pct(m1_growth, sign=True)}** |",
        f"| **영업이익 (억 원)** | {_fmt_eok(op_prev)} | **{_fmt_eok(op_curr)}** | **OPM (M2): {_fmt_pct(m2_opm)}** ({year - 1}년 {_fmt_pct(opm_prev)}) |",
        f"| **당기순이익 (억 원)** | {_fmt_eok(ni_prev)} | **{_fmt_eok(ni_curr)}** | **NPM: {_fmt_pct(npm_curr)}** ({year - 1}년 {_fmt_pct(npm_prev)}) |",
        f"| **부채총계 (억 원)** | {_fmt_eok(debt_prev)} | **{_fmt_eok(debt_curr)}** | **부채비율 (M3): {_fmt_pct(m3_debt_ratio)}** ({year - 1}년 {_fmt_pct(debt_ratio_prev)}) |",
        f"| **자본총계 (억 원)** | {_fmt_eok(eq_prev)} | **{_fmt_eok(eq_curr)}** | **자산총계: {_fmt_trillion(asset_curr)}** |",
        "",
        "---",
        "",
        "## 📈 [Page 2: KEY CHARTS] 투자포인트 논리 도식 & 동적 벤치마크 밴드",
        "```mermaid",
        "flowchart LR",
        f"    A[\"2쪽 산업분석(장수련)<br/>전방 시장 고성장 검증 PASS\"] --> D[\"1쪽 센터장(김규민) 종합 결론\"]",
        f"    B[\"3쪽 기업·재무(박수연)<br/>매출 M1 {_fmt_pct(m1_growth, sign=True)} · OPM M2 {_fmt_pct(m2_opm)} PASS\"] --> D",
        f"    C[\"4쪽 밸류에이션(조미정)<br/>현재가 {cur_price:,}원 → 목표가 {tgt_price:,}원 ({upside_pct:+.2f}%)\"] --> D",
        f"    E[\"동적 벤치마크 밴드<br/>한은 금리({rf_pct:.2f}%) ~ S&P500 10Y({sp500_pct:.2f}%)\"] --> D",
        f"    D --> F[\"최종 투자의견: {rating_str}\"]",
        "```",
        "",
        "### 🔍 [출구 ④·⑤ 검증 기록지 요약]",
        f"- **1쪽 결론 무결성 검사 통과 여부**: `{conclusion_check.get('결론검사_통과', False)}` (수록 주장 전수 `PASS` 및 조원 근거 100% 매핑 완료)",
    ]

    if rejected_claims:
        md_lines.append("- **조원 파트로 되돌린(반려) 주장 내역 (1쪽 표지 수록 차단됨)**:")
        for rc in rejected_claims:
            md_lines.append(
                f"  - `{rc['claim_id']}` ({rc.get('파트', '')}): `{rc['판정_목록']}` — {rc['탈락_이유']} ➔ *해당 파트로 수정 요청*"
            )

    markdown_text = "\n".join(md_lines) + "\n"

    # ------------------------------------------------------------------
    # 2) SMIC 농심홀딩스 보고서 2단 커버 레이아웃 HTML 생성 (코랩 출력용)
    # ------------------------------------------------------------------
    html_text = f"""
<div style="font-family: 'Malgun Gothic', 'Apple SD Gothic Neo', sans-serif; max-width: 1050px; margin: 16px auto; background: #ffffff; color: #1f2937; border: 1px solid #d1d5db; box-shadow: 0 4px 16px rgba(0,0,0,0.08); padding: 28px 32px; box-sizing: border-box;">
  <!-- 상단 SMIC 스타일 리서치 헤더 배너 -->
  <div style="display: flex; justify-content: space-between; align-items: flex-end; border-bottom: 3px solid #1b365d; padding-bottom: 14px; margin-bottom: 20px;">
    <div>
      <div style="font-size: 13px; font-weight: 700; color: #1b365d; letter-spacing: 0.5px;">
        🏛️ KFA AI RESEARCH CENTER TEAM 1 &nbsp;|&nbsp; {today_str}
      </div>
      <div style="font-size: 28px; font-weight: 900; color: #111827; margin-top: 6px; line-height: 1.2;">
        {stock_name} <span style="font-size: 20px; font-weight: 700; color: #4b5563;">({stock_code})</span>
      </div>
    </div>
    <div style="text-align: right;">
      <div style="background: #1b365d; color: #ffffff; padding: 8px 18px; border-radius: 4px; font-size: 18px; font-weight: 800; letter-spacing: -0.3px;">
        “{main_catchphrase}”
      </div>
    </div>
  </div>

  <!-- 메인 2단 레이아웃 (좌측 본문 67% + 우측 사이드바 31%) -->
  <div style="display: flex; gap: 24px; align-items: flex-start; flex-wrap: wrap;">

    <!-- [좌측 메인 컬럼] 내러티브 본문 & 하단 요약 손익계산서 -->
    <div style="flex: 1 1 620px; min-width: 320px;">
      <!-- 도입부 Executive Hook -->
      <div style="margin-bottom: 18px;">
        <div style="font-size: 16px; font-weight: 800; color: #1b365d; border-left: 4px solid #1b365d; padding-left: 10px; margin-bottom: 8px;">
          {hook_title}
        </div>
        <p style="font-size: 13.5px; line-height: 1.68; color: #374151; text-align: justify; margin: 0;">
          {hook_body}
        </p>
      </div>

      <!-- 투자포인트 1 -->
      <div style="margin-bottom: 18px;">
        <div style="font-size: 15.5px; font-weight: 800; color: #1b365d; border-left: 4px solid #2563eb; padding-left: 10px; margin-bottom: 8px;">
          {tp1_title}
        </div>
        <p style="font-size: 13.5px; line-height: 1.68; color: #374151; text-align: justify; margin: 0;">
          {tp1_body}
        </p>
      </div>

      <!-- 투자포인트 2 -->
      <div style="margin-bottom: 18px;">
        <div style="font-size: 15.5px; font-weight: 800; color: #1b365d; border-left: 4px solid #2563eb; padding-left: 10px; margin-bottom: 8px;">
          {tp2_title}
        </div>
        <p style="font-size: 13.5px; line-height: 1.68; color: #374151; text-align: justify; margin: 0;">
          {tp2_body}
        </p>
      </div>

      <!-- Valuation -->
      <div style="margin-bottom: 22px;">
        <div style="font-size: 15.5px; font-weight: 800; color: #1b365d; border-left: 4px solid #d9381e; padding-left: 10px; margin-bottom: 8px;">
          Valuation: 목표주가 {tgt_price:,}원, 최종 투자의견 {rating_str}
        </div>
        <p style="font-size: 13.5px; line-height: 1.68; color: #374151; text-align: justify; margin: 0;">
          {val_body}
        </p>
      </div>

      <!-- 하단 연결포괄손익계산서 요약표 (SMIC 1쪽 하단 테이블 스타일) -->
      <div style="margin-top: 14px;">
        <div style="font-size: 13px; font-weight: 800; color: #1b365d; margin-bottom: 6px;">
          ■ 연결포괄손익계산서 및 핵심 재무지표 요약 (단위: 억 원, %)
        </div>
        <table style="width: 100%; border-collapse: collapse; font-size: 12.5px; text-align: right; border-top: 2px solid #1b365d; border-bottom: 2px solid #1b365d;">
          <thead>
            <tr style="background: #f3f6fa; color: #1b365d; border-bottom: 1px solid #cbd5e1;">
              <th style="padding: 7px 10px; text-align: left;">구분 (연결)</th>
              <th style="padding: 7px 10px; text-align: right;">{year - 1}년</th>
              <th style="padding: 7px 10px; text-align: right;">{year}년</th>
              <th style="padding: 7px 10px; text-align: right;">핵심 검증 지표 (M1~M3)</th>
            </tr>
          </thead>
          <tbody>
            <tr style="border-bottom: 1px solid #e5e7eb;">
              <td style="padding: 6px 10px; text-align: left; font-weight: 700;">매출액 (억 원)</td>
              <td style="padding: 6px 10px;">{_fmt_eok(rev_prev)}</td>
              <td style="padding: 6px 10px; font-weight: 700;">{_fmt_eok(rev_curr)}</td>
              <td style="padding: 6px 10px; color: #1d4ed8; font-weight: 700;">YoY(M1): {_fmt_pct(m1_growth, sign=True)}</td>
            </tr>
            <tr style="border-bottom: 1px solid #e5e7eb; background: #fafafa;">
              <td style="padding: 6px 10px; text-align: left; font-weight: 700;">영업이익 (억 원)</td>
              <td style="padding: 6px 10px;">{_fmt_eok(op_prev)}</td>
              <td style="padding: 6px 10px; font-weight: 700;">{_fmt_eok(op_curr)}</td>
              <td style="padding: 6px 10px; color: #1d4ed8; font-weight: 700;">OPM(M2): {_fmt_pct(m2_opm)}</td>
            </tr>
            <tr style="border-bottom: 1px solid #e5e7eb;">
              <td style="padding: 6px 10px; text-align: left;">당기순이익 (억 원)</td>
              <td style="padding: 6px 10px;">{_fmt_eok(ni_prev)}</td>
              <td style="padding: 6px 10px; font-weight: 700;">{_fmt_eok(ni_curr)}</td>
              <td style="padding: 6px 10px;">NPM: {_fmt_pct(npm_curr)}</td>
            </tr>
            <tr>
              <td style="padding: 6px 10px; text-align: left;">부채총계 / 자본총계 (억 원)</td>
              <td style="padding: 6px 10px;">{_fmt_eok(debt_prev)} / {_fmt_eok(eq_prev)}</td>
              <td style="padding: 6px 10px; font-weight: 700;">{_fmt_eok(debt_curr)} / {_fmt_eok(eq_curr)}</td>
              <td style="padding: 6px 10px; color: #047857; font-weight: 700;">부채비율(M3): {_fmt_pct(m3_debt_ratio)}</td>
            </tr>
          </tbody>
        </table>
      </div>
    </div>

    <!-- [우측 사이드바] SMIC 스타일 투자의견 & Key Data 박스 -->
    <div style="flex: 0 0 295px; background: #f8fafc; border: 1px solid #cbd5e1; border-top: 4px solid {badge_bg}; padding: 16px; box-sizing: border-box;">
      <!-- Rating 박스 -->
      <div style="border-bottom: 2px solid #cbd5e1; padding-bottom: 12px; margin-bottom: 14px;">
        <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px;">
          <span style="font-size: 13px; font-weight: 700; color: #475569;">투자의견 (Rating)</span>
          <span style="background: {badge_accent}; color: #ffffff; font-size: 16px; font-weight: 900; padding: 4px 12px; border-radius: 4px;">
            {rating_str}
          </span>
        </div>
        <div style="display: flex; justify-content: space-between; font-size: 13px; margin-bottom: 6px;">
          <span style="color: #64748b;">현재주가</span>
          <span style="font-weight: 800; color: #111827;">{cur_price:,} 원</span>
        </div>
        <div style="display: flex; justify-content: space-between; font-size: 13px; margin-bottom: 6px;">
          <span style="color: #64748b;">목표주가 (4쪽 밸류)</span>
          <span style="font-weight: 800; color: #d9381e;">{tgt_price:,} 원</span>
        </div>
        <div style="display: flex; justify-content: space-between; font-size: 13px;">
          <span style="color: #64748b;">상승여력 (기대수익률)</span>
          <span style="font-weight: 900; color: #1d4ed8;">{upside_pct:+.2f}%</span>
        </div>
      </div>

      <!-- 센터장 동적 벤치마크 밴드 (Rf ~ S&P 500) -->
      <div style="border-bottom: 2px solid #cbd5e1; padding-bottom: 12px; margin-bottom: 14px;">
        <div style="font-size: 12.5px; font-weight: 800; color: #1b365d; margin-bottom: 8px;">
          ⚖️ 동적 벤치마크 밴드 (센터장 모델)
        </div>
        <div style="display: flex; justify-content: space-between; font-size: 12.5px; margin-bottom: 5px;">
          <span style="color: #64748b;">하단 (한은 기준금리 R<sub>f</sub>)</span>
          <span style="font-weight: 700;">{rf_pct:.2f}%</span>
        </div>
        <div style="display: flex; justify-content: space-between; font-size: 12.5px; margin-bottom: 5px;">
          <span style="color: #64748b;">상단 (S&amp;P 500 10Y 평균)</span>
          <span style="font-weight: 700;">{sp500_pct:.2f}%</span>
        </div>
        <div style="display: flex; justify-content: space-between; font-size: 12.5px; background: #e2e8f0; padding: 4px 8px; border-radius: 4px; margin-top: 6px;">
          <span style="font-weight: 700; color: #334155;">보유(HOLD) 구간</span>
          <span style="font-weight: 800; color: #1b365d;">{rating_decision['보유_HOLD_구간']}</span>
        </div>
      </div>

      <!-- Key Metrics -->
      <div style="border-bottom: 2px solid #cbd5e1; padding-bottom: 12px; margin-bottom: 14px;">
        <div style="font-size: 12.5px; font-weight: 800; color: #1b365d; margin-bottom: 8px;">
          📌 Key Metrics ({year}년 기준)
        </div>
        <div style="display: flex; justify-content: space-between; font-size: 12.5px; margin-bottom: 5px;">
          <span style="color: #64748b;">매출성장률 (M1)</span>
          <span style="font-weight: 700;">{_fmt_pct(m1_growth, sign=True)}</span>
        </div>
        <div style="display: flex; justify-content: space-between; font-size: 12.5px; margin-bottom: 5px;">
          <span style="color: #64748b;">영업이익률 (M2)</span>
          <span style="font-weight: 700;">{_fmt_pct(m2_opm)}</span>
        </div>
        <div style="display: flex; justify-content: space-between; font-size: 12.5px; margin-bottom: 5px;">
          <span style="color: #64748b;">밸류에이션 멀티플</span>
          <span style="font-weight: 700;">{target_mult_str}</span>
        </div>
        <div style="display: flex; justify-content: space-between; font-size: 12.5px;">
          <span style="color: #64748b;">S&amp;P 500 대비 알파</span>
          <span style="font-weight: 700; color: #1d4ed8;">{alpha_vs_sp500:+.2f}%p</span>
        </div>
      </div>

      <!-- B/S Data -->
      <div style="border-bottom: 2px solid #cbd5e1; padding-bottom: 12px; margin-bottom: 14px;">
        <div style="font-size: 12.5px; font-weight: 800; color: #1b365d; margin-bottom: 8px;">
          🏦 B/S Data ({str(year)[2:]}년 연결)
        </div>
        <div style="display: flex; justify-content: space-between; font-size: 12.5px; margin-bottom: 5px;">
          <span style="color: #64748b;">자산 총계</span>
          <span style="font-weight: 700;">{_fmt_trillion(asset_curr)}</span>
        </div>
        <div style="display: flex; justify-content: space-between; font-size: 12.5px; margin-bottom: 5px;">
          <span style="color: #64748b;">부채 총계 (부채비율 M3)</span>
          <span style="font-weight: 700;">{_fmt_trillion(debt_curr)} ({_fmt_pct(m3_debt_ratio)})</span>
        </div>
        <div style="display: flex; justify-content: space-between; font-size: 12.5px;">
          <span style="color: #64748b;">자본 총계</span>
          <span style="font-weight: 700;">{_fmt_trillion(eq_curr)}</span>
        </div>
      </div>

      <!-- KFA 1조 리서치 센터 -->
      <div style="font-size: 12px; line-height: 1.6; color: #334155;">
        <div style="font-weight: 800; color: #1b365d; margin-bottom: 4px;">👑 KFA 1조 리서치 센터</div>
        <div>• <b>센터장(1쪽·출구④⑤)</b>: 김규민</div>
        <div>• <b>산업분석(2쪽)</b>: 장수련</div>
        <div>• <b>기업·재무(3쪽)</b>: 박수연</div>
        <div>• <b>밸류에이션(4쪽)</b>: 조미정</div>
        <div>• <b>검증관·엔진</b>: 채주성 · 박형민</div>
      </div>
    </div>

  </div>
</div>
"""

    filled_prompt = build_smic_prompt_for_stock(
        stock_code=stock_code,
        stock_name=stock_name,
        year=year,
        tagged_records=tagged_records,
        metrics_df=metrics_df,
        sources_df=sources_df,
        rating_decision=rating_decision,
        page1_claims=page1_claims,
        conclusion_check=conclusion_check,
        rejected_claims=rejected_claims,
    )

    return {
        "markdown": markdown_text,
        "html": html_text,
        "prompt": filled_prompt,
    }

print("✅ 센터장 1쪽 범용 에이전트 도구(chief_agent.py) 준비 완료")

✅ 센터장 1쪽 범용 에이전트 도구(chief_agent.py) 준비 완료


In [55]:
# 🔑 5. API 키 불러오기 (왼쪽 🔑 보안 비밀에서 자동으로 읽어와요)
DART_API_KEY = get_key("DART_API_KEY")
ECOS_API_KEY = get_key("ECOS_API_KEY")
FRED_API_KEY = get_key("FRED_API_KEY")
GEMINI_API_KEY = get_key("GEMINI_API_KEY")

print("DART_API_KEY  :", "등록됨 ✅" if DART_API_KEY else "없음 ⏭ (다른 종목을 넣으려면 DART_API_KEY가 꼭 필요해요!)")
print("ECOS_API_KEY  :", "등록됨 ✅" if ECOS_API_KEY else "없음 ⏭ (없으면 FRED 한국금리 시계열로 자동 대체돼요)")
print("FRED_API_KEY  :", "등록됨 ✅" if FRED_API_KEY else "없음 ⏭ (등록됨 -> 미국 연준 FRED 공식 API로 S&P 500 10년치 수신!)")
print("GEMINI_API_KEY:", "등록됨 ✅" if GEMINI_API_KEY else "없음 ⏭")

DART_API_KEY  : 등록됨 ✅
ECOS_API_KEY  : 등록됨 ✅
FRED_API_KEY  : 등록됨 ✅
GEMINI_API_KEY: 등록됨 ✅


## Step 1. 🤝 [출구 ④] 조원들(산업·기업·밸류에이션)이 작업한 파일·주장 넣기 & `YEAR` / `TARGET_PRICE` 자동 감지

조원들이 보내준 CSV 파일(`TEAMMATE_CSV_FILES`)이나 초안 문장(`TEAMMATE_CLAIMS`, `TEAMMATE_DRAFT_TEXTS`)을 여기에 넣으면,
센터장 에이전트의 **`auto_detect_config_from_teammates()`** 함수가 조원들의 결과물을 스캔해서 **종목코드(`STOCK_CODE`), 기준연도(`YEAR`), 4쪽 목표주가(`TARGET_PRICE`)를 자동으로 찾아냅니다!**

In [56]:
# 🔍 0. 조원 CSV 파일 자동 스캔 함수 (폴더에 있는 파일을 스스로 찾음)
import glob
from pathlib import Path
import pandas as pd

def find_teammate_csv_files(search_dir="."):
    discovered = []
    for p in Path(search_dir).glob("*.csv"):
        name = p.name.lower()
        if any(k in name for k in ["산업", "기업", "재무", "밸류", "w03", "w04"]):
            discovered.append(str(p))
            continue
        try:
            head = pd.read_csv(p, nrows=2)
            if {"종목코드", "항목", "값"}.issubset(set(head.columns)):
                discovered.append(str(p))
        except Exception:
            pass
    return sorted(list(dict.fromkeys(discovered)))


# 🔍 1. 코랩 폴더에 있는 조원들의 CSV 파일을 스스로 찾아서 연결!
auto_found_files = find_teammate_csv_files(".")

# 만약 폴더에 파일이 하나도 없으면 농심홀딩스 예시 파일을 자동 생성
if not auto_found_files:
    print("📁 업로드된 파일이 없어 농심홀딩스 예시 파일을 생성합니다...")
    sample_bundle = create_sample_teammate_files(".", stock_code="072710", stock_name="농심홀딩스", year=2024)
    TEAMMATE_CSV_FILES = sample_bundle["csv_files"]
    TEAMMATE_CLAIMS = sample_bundle["claims"]
else:
    TEAMMATE_CSV_FILES = auto_found_files
    TEAMMATE_CLAIMS = []

print(f"📥 [조원 CSV 파일 {len(TEAMMATE_CSV_FILES)}개 자동 발견 및 연결 완료!]")
for f_path in TEAMMATE_CSV_FILES:
    df_p = pd.read_csv(f_path)
    s_name = df_p["종목명"].dropna().iloc[0] if "종목명" in df_p.columns and not df_p["종목명"].dropna().empty else "미지정"
    print(f"   📄 {Path(f_path).name} ({len(df_p)}줄, 종목: {s_name})")

TEAMMATE_METRIC_SPECS = []
TEAMMATE_DRAFT_TEXTS = {
    "2쪽_산업분석(장수련)": "",
    "3쪽_기업재무(박수연)": "",
    "4쪽_밸류에이션(조미정)": "",
}

# 🔍 2. 발견된 조원 파일들에서 종목코드, 기준연도, 목표주가 자동 판별!
auto_cfg = auto_detect_config_from_teammates(
    stock_code=STOCK_CODE,
    year=YEAR,
    target_price=TARGET_PRICE,
    teammate_csv_paths=TEAMMATE_CSV_FILES,
    teammate_claims=TEAMMATE_CLAIMS,
    teammate_draft_texts=TEAMMATE_DRAFT_TEXTS,
    dart_api_key=DART_API_KEY,
)
STOCK_CODE = auto_cfg["STOCK_CODE"]
YEAR = auto_cfg["YEAR"]
TARGET_PRICE = auto_cfg["TARGET_PRICE"]

print("\n🤖 [센터장 에이전트: 조원 파일에서 자동 감지 완료]")
for log_msg in auto_cfg["감지_로그"]:
    print("  ✔", log_msg)
print(f"  👉 최종 적용: 종목코드={STOCK_CODE}, 기준연도={YEAR}년, 목표주가={int(TARGET_PRICE):,}원")

📥 [조원 CSV 파일 3개 자동 발견 및 연결 완료!]
   📄 w03_박수연_기업분석.csv (9줄, 종목: 삼성전자)
   📄 w03_장수련_산업분석.csv (2줄, 종목: 삼성전자)
   📄 w03_조미정_밸류에이션.csv (4줄, 종목: 삼성전자)

🤖 [센터장 에이전트: 조원 파일에서 자동 감지 완료]
  ✔ 종목코드 자동 감지(조원 CSV): 005930
  ✔ 기준연도(YEAR) 자동 감지(조원 CSV 기준일): 2026년
  ✔ 목표주가(TARGET_PRICE) 자동 감지(4쪽 밸류에이션 조원 표): 250,000원
  👉 최종 적용: 종목코드=005930, 기준연도=2026년, 목표주가=250,000원


## Step 1.5. 자동 감지된 종목(`STOCK_CODE`)·연도(`YEAR`)·목표주가(`TARGET_PRICE`)로 DART + 주가 + 한은 금리 + S&P 500 수집

In [57]:
# 1) 조원 파일에 이미 기업재무 데이터가 있는지 확인하고, DART_API_KEY가 있으면 추가로 검증용 수집
dart_years = []
if DART_API_KEY:
    for y in (YEAR - 1, YEAR):
        res = run_step(f"DART {STOCK_CODE} {y} 사업보고서", 1, lambda y=y: collect_dart(STOCK_CODE, y, 4, "CFS", DART_API_KEY))
        if res is not None:
            dart_years.append(res)

# 회사 이름(STOCK_NAME)은 조원 CSV 파일 또는 DART에서 자동으로 읽어와요
STOCK_NAME = f"종목_{STOCK_CODE}"
if dart_years:
    STOCK_NAME = str(dart_years[-1]["종목명"].iloc[0])
elif TEAMMATE_CSV_FILES:
    for f_p in TEAMMATE_CSV_FILES:
        if Path(f_p).exists():
            t_df = pd.read_csv(f_p)
            if "종목명" in t_df.columns and not t_df["종목명"].dropna().empty:
                STOCK_NAME = str(t_df["종목명"].dropna().iloc[0])
                break

# 2) 센터장 에이전트 전용 실시간 벤치마크 API: 한국은행 기준금리(무위험이자율) & S&P 500 10년 연평균 수익률(FRED_API_KEY)
rec_bok = run_step("한국은행 기준금리(무위험이자율) 실시간 API", 1, lambda: collect_bok_rf_rate_records(ECOS_API_KEY))
rec_sp500 = run_step("S&P 500 10년 연평균 수익률 실시간 API", 1, lambda: collect_sp500_10y_cagr_records(FRED_API_KEY))

# 3) 만약 4쪽 밸류에이션 조원 파일이 없을 때만 실시간 주가·목표주가 수집기를 보조로 돌려요
has_teammate_val = any("밸류" in str(p) for p in TEAMMATE_CSV_FILES if Path(p).exists())
rec_price = None
if not has_teammate_val:
    rec_price = run_step(
        f"{STOCK_NAME}({STOCK_CODE}) 현재주가 및 목표주가",
        2,
        lambda: collect_stock_price_and_target_records(STOCK_CODE, STOCK_NAME, TARGET_PRICE, f"{YEAR}-12-31"),
    )

all_records = pd.concat([f for f in (*dart_years, rec_bok, rec_sp500, rec_price) if f is not None], ignore_index=True)
print(f"✅ 센터장 거시 벤치마크(한은 기준금리 + S&P500 10년 평균) 준비 완료 ({len(all_records)}줄)")

✅ DART 005930 2025 사업보고서: 6줄 (확인 필요 0줄)
❌ DART 005930 2026 사업보고서: DART 013: 조회된 데이터가 없어요. 아직 공시 전인 분기인지 확인해 주세요
✅ 한국은행 기준금리(무위험이자율) 실시간 API: 1줄 (확인 필요 0줄)
✅ S&P 500 10년 연평균 수익률 실시간 API: 3줄 (확인 필요 0줄)
✅ 센터장 거시 벤치마크(한은 기준금리 + S&P500 10년 평균) 준비 완료 (10줄)


## Step 2. 조원 데이터 통합(`merge_teammates_tables_and_metrics`) · 줄 이름표(`rid`) · 출처 번호(`[S#]`) · 지표 계산(`[M#]`)

센터장의 첫 번째 핵심 역할(**출구 ④ 파트 결과 모으기**)입니다!
기본 데이터와 조원들이 보내준 표를 하나로 합치면서:
1. 조원들이 중복해서 수집한 같은 사실(`rid`)이 있으면 충돌이 나지 않게 하나로 정리하고,
2. 전체 합본 표 기준으로 출처 번호(`[S1], [S2]...`)를 **딱 한 번에 통일**해서 매기며,
3. 기본 지표(`M1~M3`)와 조원 추가 지표(`TEAMMATE_METRIC_SPECS`)를 한꺼번에 계산합니다.

In [58]:
sources_df, tagged_records, metrics_df = merge_teammates_tables_and_metrics(
    base_records=all_records,
    stock_code=STOCK_CODE,
    stock_name=STOCK_NAME,
    year=YEAR,
    teammate_csv_paths=TEAMMATE_CSV_FILES,
    teammate_metric_specs=TEAMMATE_METRIC_SPECS,
)

print(f"📌 [{STOCK_NAME}({STOCK_CODE}) 통합 출처 목록 S# (조원 데이터 포함)]")
display(sources_df)
print(f"📌 [{STOCK_NAME}({STOCK_CODE}) 통합 계산된 지표 표 M#]")
display(metrics_df)

📥 조원 CSV 로드 완료: w03_박수연_기업분석.csv (9줄)
📥 조원 CSV 로드 완료: w03_장수련_산업분석.csv (2줄)
📥 조원 CSV 로드 완료: w03_조미정_밸류에이션.csv (4줄)
📌 [삼성전자(005930) 통합 출처 목록 S# (조원 데이터 포함)]


,source_id,출처,출처 URL,수집 방법,신뢰도,기준일 범위,줄 수
0,S1,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/005930_2023,1 공식 API,상,2023-12-31,4
1,S2,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/005930_2024,1 공식 API,상,2024-12-31,5
2,S3,DART 단일회사 전체 재무제표 API,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,1 공식 API,상,2025-12-31,6
3,S4,FRED Open API (S&P 500 공식 지수),https://fred.stlouisfed.org/series/SP500,1 공식 API,상,2016-11-14 ~ 2026-11-10,3
4,S5,FinanceDataReader (KRX 확정종가 및 밸류에이션),https://finance.naver.com/item/main.naver?code...,2 파이썬 라이브러리,중,2026-11-11,4
5,S6,IDC 및 Gartner 공식 통계 (2쪽 산업분석 장수련),https://www.gartner.com/foundry2026,1 공식 API,상,2026-11-11,1
6,S7,IDC 및 Gartner 공식 통계 (2쪽 산업분석 장수련),https://www.idc.com/semiconductor2026,1 공식 API,상,2026-11-11,1
7,S8,한국은행 기준금리 (ECOS / FRED INTDSRKRM193N 실시간),https://fred.stlouisfed.org/series/INTDSRKRM193N,3 파일 다운로드,상,2026-07-01,1


📌 [삼성전자(005930) 통합 계산된 지표 표 M#]


,metric_id,지표,값,단위,계산식,입력 rid,source_id,기준일,상태,이유
0,M1,매출 성장률,None,%,(올해 매출액(연간) − 작년 매출액(연간)) ÷ 작년 매출액(연간) × 100,[005930|매출액(연간)|연결|2025-12-31|DART 단일회사 전체 재무제...,[S3],2026-12-31,확인 필요,매출액(연간) 2026-12-31: 연결 줄 없음
1,M2,영업이익률,None,%,영업이익(연간) ÷ 매출액(연간) × 100,[],[],2026-12-31,확인 필요,영업이익(연간) 2026-12-31: 연결 줄 없음; 매출액(연간) 2026-12-...
2,M3,부채비율,None,%,부채총계 ÷ 자본총계 × 100,[],[],2026-12-31,확인 필요,부채총계 2026-12-31: 연결 줄 없음; 자본총계 2026-12-31: 연결 ...


## Step 3. 동적 벤치마크(현 한은 기준금리 ~ S&P 500 10년 평균 수익률) 기반 투자의견(`BUY / HOLD / SELL`) 자동 판정

4쪽 밸류에이션 담당 조원(조미정님)이 산출한 `TARGET_PRICE(목표주가)`를 받아와서, 센터장의 실시간 동적 밴드(`현 한국은행 기준금리 ~ S&P 500 10년 연평균 수익률`)에 대입해 최종 투자의견(`BUY / HOLD / SELL`)을 판정합니다.

In [59]:
bok_row = tagged_records[(tagged_records["종목코드"] == "BOK") & (tagged_records["항목"] == "한국은행기준금리(무위험이자율)")].iloc[0]
sp500_row = tagged_records[(tagged_records["종목코드"] == "SP500") & (tagged_records["항목"] == "10년연평균수익률(CAGR)")].iloc[0]
cur_p_row = tagged_records[(tagged_records["종목코드"] == STOCK_CODE) & (tagged_records["항목"] == "현재주가(확정종가)")].iloc[0]
tgt_p_row = tagged_records[(tagged_records["종목코드"] == STOCK_CODE) & (tagged_records["항목"] == "목표주가(Base)")].iloc[0]

bok_rf_val = float(bok_row["값"])
sp500_cagr_val = float(sp500_row["값"])
cur_p_val = float(cur_p_row["값"])
tgt_p_val = float(tgt_p_row["값"])

rating_decision = decide_rating_by_dynamic_band(
    current_price=cur_p_val,
    target_price=tgt_p_val,
    bok_rf_rate_pct=bok_rf_val,
    sp500_10y_cagr_pct=sp500_cagr_val,
)

print(f"🎯 [{STOCK_NAME}({STOCK_CODE}) 실시간 동적 밴드 판정]:")
print(f"   하단(한은 기준금리 {bok_rf_val:.2f}%) ~ 상단(S&P500 10년 평균 {sp500_cagr_val:.2f}%) | 기대수익률: {rating_decision['기대수익률_pct']:.2f}% ➔ {rating_decision['투자의견']}")
display(pd.DataFrame([rating_decision]))

🎯 [삼성전자(005930) 실시간 동적 밴드 판정]:
   하단(한은 기준금리 1.25%) ~ 상단(S&P500 10년 평균 13.28%) | 기대수익률: 25.69% ➔ BUY (매수)


,현재주가,목표주가,기대수익률_pct,하단기준_한국은행기준금리_Rf_pct,상단기준_SP500_10년평균_pct,보유_HOLD_구간,SP500대비_초과수익률_pct,무위험이자율대비_초과수익률_pct,투자의견,판정_근거
0,198900,250000,25.69,1.25,13.28,1.25% ~ 13.28%,12.41,24.44,BUY (매수),기대수익률(25.69%)이 상단 기준선인 S&P 500 10년 연평균 수익률(13....


## Step 4. 조원 주장 검증(`PASS` 필터링 & `FIX/REVIEW` 반려) 및 1쪽 리포트 합성 (`synthesize_teammates_to_page1`)

센터장의 두 번째 핵심 역할(**출구 ⑤ 결론 내기**)입니다!
- 조원들이 넘겨준 2~4쪽 주장들을 `check_claims()`로 전수 검사해서:
  - **`FIX`(숫자 틀림)나 `REVIEW`(근거 누락)**가 나온 주장은 1쪽 표지에 절대 올리지 않고 **해당 조원에게 반려(되돌림)** 목록으로 정리합니다.
  - **`PASS`(원본 대조 통과)**를 받은 조원들의 핵심 주장만 모아서 센터장의 투자의견(`BUY / HOLD / SELL`)과 함께 **1쪽 투자 요약 리포트**로 완성합니다!

In [60]:
try:
    from IPython.display import Markdown, HTML
except ImportError:
    Markdown = lambda x: x
    HTML = lambda x: x

synthesis = synthesize_teammates_to_page1(
    stock_code=STOCK_CODE,
    stock_name=STOCK_NAME,
    year=YEAR,
    tagged_records=tagged_records,
    metrics_df=metrics_df,
    sources_df=sources_df,
    rating_decision=rating_decision,
    teammate_claims=TEAMMATE_CLAIMS,
    teammate_draft_texts=TEAMMATE_DRAFT_TEXTS,
    gemini_api_key=GEMINI_API_KEY,
)

checked_df = synthesis["checked_df"]
passed_claims = synthesis["passed_claims"]
rejected_claims = synthesis["rejected_claims"]
page1_claims = synthesis["page1_claims"]
conclusion_check = synthesis["conclusion_check"]

print(f"📌 [1] 조원(2~4쪽) 주장 숫자 대조 결과 (총 {len(checked_df)}건 대조)")
display(checked_df)

if rejected_claims:
    print(f"🚨 [2] 조원 파트로 되돌릴(반려) 주장 목록 ({len(rejected_claims)}건 — 1쪽 표지 수록 차단됨!)")
    display(pd.DataFrame(rejected_claims)[["claim_id", "파트", "문장", "판정_목록", "탈락_이유"]])

print("✅ [3] 출구 ⑤ 1쪽 결론 무결성 검사 통과 여부:", conclusion_check["결론검사_통과"])
display(conclusion_check["p1_checked_df"])

# 🏛️ SMIC(서울대 투자연구회) '농심홀딩스(072710)' 보고서 1쪽 양식(2단 커버 레이아웃 + 요약 재무제표 + KEY CHARTS) 렌더링
smic_pack = render_smic_style_page1_report(
    stock_code=STOCK_CODE,
    stock_name=STOCK_NAME,
    year=YEAR,
    tagged_records=tagged_records,
    metrics_df=metrics_df,
    sources_df=sources_df,
    rating_decision=rating_decision,
    page1_claims=page1_claims,
    conclusion_check=conclusion_check,
    rejected_claims=rejected_claims,
)

report_file = Path(f"w05_page1_{STOCK_CODE}_{YEAR}.md")
report_file.write_text(smic_pack["markdown"], encoding="utf-8")

prompt_file = Path(f"w05_chief_smic_prompt_{STOCK_CODE}_{YEAR}.txt")
prompt_file.write_text(smic_pack["prompt"], encoding="utf-8")
print(f"💾 SMIC 포맷 1쪽 마크다운 저장 완료: {report_file}")
print(f"💾 센터장 SMIC 포맷 전용 프롬프트 저장 완료: {prompt_file}")

# 코랩 화면에 SMIC 보고서 1쪽과 동일한 2단 커버 레이아웃(HTML) 출력!
display(HTML(smic_pack["html"]))

📌 [1] 조원(2~4쪽) 주장 숫자 대조 결과 (총 5건 대조)


,claim_id,문장,근거 id,적은 숫자,적은 단위,실제 값,판정,이유,출처 번호
0,C-3,"삼성전자의 현재주가 198,900원 대비 목표주가는 250,000원으로 기대수익률은...",005930|현재주가(확정종가)|해당 없음|2026-11-11|FinanceData...,"198,900",원,198900.00,PASS,,[S4][S5][S8]
1,C-3,"삼성전자의 현재주가 198,900원 대비 목표주가는 250,000원으로 기대수익률은...",005930|목표주가(Base)|해당 없음|2026-11-11|FinanceData...,"250,000",원,250000.00,PASS,,[S4][S5][S8]
2,C-3,"삼성전자의 현재주가 198,900원 대비 목표주가는 250,000원으로 기대수익률은...",005930|목표주가기대수익률|해당 없음|2026-11-11|FinanceDataR...,25.70,%,25.70,PASS,,[S4][S5][S8]
3,C-3,"삼성전자의 현재주가 198,900원 대비 목표주가는 250,000원으로 기대수익률은...",BOK|한국은행기준금리(무위험이자율)|해당 없음|2026-07-01|한국은행 기준금...,1.25,%,1.25,PASS,,[S4][S5][S8]
4,C-3,"삼성전자의 현재주가 198,900원 대비 목표주가는 250,000원으로 기대수익률은...",SP500|10년연평균수익률(CAGR)|해당 없음|2026-11-10|FRED Op...,13.28,%,13.28,PASS,,[S4][S5][S8]


✅ [3] 출구 ⑤ 1쪽 결론 무결성 검사 통과 여부: True


,claim_id,문장,근거 id,적은 숫자,적은 단위,실제 값,판정,이유,출처 번호
0,P1-1,"삼성전자의 현재주가 198,900원 대비 목표주가는 250,000원으로 기대수익률은...",005930|현재주가(확정종가)|해당 없음|2026-11-11|FinanceData...,"198,900",원,198900.00,PASS,,[S4][S5][S8]
1,P1-1,"삼성전자의 현재주가 198,900원 대비 목표주가는 250,000원으로 기대수익률은...",005930|목표주가(Base)|해당 없음|2026-11-11|FinanceData...,"250,000",원,250000.00,PASS,,[S4][S5][S8]
2,P1-1,"삼성전자의 현재주가 198,900원 대비 목표주가는 250,000원으로 기대수익률은...",005930|목표주가기대수익률|해당 없음|2026-11-11|FinanceDataR...,25.70,%,25.70,PASS,,[S4][S5][S8]
3,P1-1,"삼성전자의 현재주가 198,900원 대비 목표주가는 250,000원으로 기대수익률은...",BOK|한국은행기준금리(무위험이자율)|해당 없음|2026-07-01|한국은행 기준금...,1.25,%,1.25,PASS,,[S4][S5][S8]
4,P1-1,"삼성전자의 현재주가 198,900원 대비 목표주가는 250,000원으로 기대수익률은...",SP500|10년연평균수익률(CAGR)|해당 없음|2026-11-10|FRED Op...,13.28,%,13.28,PASS,,[S4][S5][S8]


💾 SMIC 포맷 1쪽 마크다운 저장 완료: w05_page1_005930_2026.md
💾 센터장 SMIC 포맷 전용 프롬프트 저장 완료: w05_chief_smic_prompt_005930_2026.txt


구분 (연결),2025년,2026년,핵심 검증 지표 (M1~M3)
매출액 (억 원),"3,336,059",-,YoY(M1): -
영업이익 (억 원),"436,011",-,OPM(M2): -
당기순이익 (억 원),-,-,NPM: -
부채총계 / 자본총계 (억 원),"1,306,218 / 4,363,203",- / -,부채비율(M3): -
